# Fetching Data from the API

In [ ]:
from pycoingecko import CoinGeckoAPI
import pandas as pd
import time

cg = CoinGeckoAPI()

#Ambil top 20 coin berdasarkan market cap
top_20 = cg.get_coins_markets(vs_currency='usd', per_page=21, page=1)
coin_ids = [coin['id'] for coin in top_20]

#Fungsi untuk ambil data harga historis
def get_price_history(coin_id):
    try:
        data = cg.get_coin_market_chart_by_id(id=coin_id, vs_currency='usd', days=365)  # 1 tahun ke belakang dilakukan pada 2 Oktober 2025
        prices = data['prices']
        df = pd.DataFrame(prices, columns=['timestamp', coin_id])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df.set_index('timestamp', inplace=True)
        return df
    except Exception as e:
        print(f"Failed to get {coin_id}: {e}")
        return None

#Download satu per satu
price_dfs = []
for coin_id in coin_ids:
    df = get_price_history(coin_id)
    if df is not None:
        price_dfs.append(df)
    time.sleep(1)  # hindari rate limit

#Gabungkan semua harga ke satu DataFrame
prices = pd.concat(price_dfs, axis=1)
prices = prices.resample('1D').last().dropna(axis=1)  # Resample ke harian

#Hitung return harian
returns = prices.pct_change().dropna()

#Tampilkan data
print(prices.head())
print(returns.head())

# HRP

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.stats import skew, kurtosis

# =======================================
# 1. BACA DATA
# =======================================
#prices = pd.read_csv('crypto_prices.csv', index_col=0, parse_dates=True)

# Hitung return biasa (bukan log return)
returns = prices.pct_change().dropna()

print("Data return shape:", returns.shape)
print(returns.head())

# =======================================
# 2. DEFINISI FUNGSI HRP PURE
# =======================================
def correl_dist(corr):
    return np.sqrt(0.5 * (1 - corr))

def get_cluster_var(cov_df, items):
    sub_cov = cov_df.loc[items, items].values
    inv_diag = 1 / np.diag(sub_cov)
    weights = inv_diag / np.sum(inv_diag)
    return weights.T @ sub_cov @ weights

def recursive_bisection(cov_df, ordered_tickers):
    weights = pd.Series(1.0, index=ordered_tickers)
    clusters = [ordered_tickers]
    while clusters:
        new_clusters = []
        for cluster in clusters:
            if len(cluster) <= 1:
                continue
            split = len(cluster) // 2
            cluster1 = cluster[:split]
            cluster2 = cluster[split:]
            var1 = get_cluster_var(cov_df, cluster1)
            var2 = get_cluster_var(cov_df, cluster2)
            alpha = 1 - var1 / (var1 + var2)
            for asset in cluster1:
                weights[asset] *= alpha
            for asset in cluster2:
                weights[asset] *= (1 - alpha)
            new_clusters.extend([cluster1, cluster2])
        clusters = new_clusters
    return weights

# =======================================
# 3. ROLLING WINDOW SETUP (1 HARI SEKALI) - DIPERBAIKI
# =======================================
window_train = 254  # (255-1)
step = 1            # rebalancing setiap 1 hari
portfolio_values = []
weights_all = []
portfolio_returns_list = []  # Simpan returns untuk analisis

print(f"Starting Rolling HRP with {step}-day rebalancing...")

for t in range(0, 110, step):
    print(f"Processing window starting at day {t}")
    
    train = returns.iloc[t:t+window_train]
    test = returns.iloc[t+window_train:t+window_train+step]
    
    if test.empty or len(train) < window_train:
        print(f"Skipping window {t} - insufficient data")
        continue

    try:
        # --- HRP di training window ---
        corr_matrix = train.corr()
        cov_matrix = train.cov()
        cov_df = pd.DataFrame(cov_matrix, index=train.columns, columns=train.columns)

        # --- Hierarchical clustering ---
        dist = correl_dist(corr_matrix)
        link = linkage(dist, method='single')
        ordered_idx = leaves_list(link)
        ordered_tickers = list(train.columns[ordered_idx])

        # --- Hitung bobot HRP ---
        weights = recursive_bisection(cov_df, ordered_tickers)
        weights /= weights.sum()
        weights_all.append(weights)

        # --- Hitung return portofolio pada hari test ---
        for i in range(len(test)):
            port_ret = test.iloc[i] @ weights
            portfolio_returns_list.append(port_ret)
            
            if len(portfolio_values) == 0:
                port_val = 1.0 * (1 + port_ret)
            else:
                port_val = portfolio_values[-1] * (1 + port_ret)
                
            portfolio_values.append(port_val)
            
    except Exception as e:
        print(f"Error in window {t}: {e}")
        continue

# =======================================
# 4. GABUNG HASIL & VISUALISASI PORTFOLIO - DIPERBAIKI
# =======================================
if portfolio_values:
    # Buat time series portfolio values
    final_portfolio = pd.Series(portfolio_values, 
                               index=returns.index[window_train:window_train + len(portfolio_values)])
    
    plt.figure(figsize=(12,6))
    plt.plot(final_portfolio.index, final_portfolio.values, 
             label="Rolling HRP Portfolio (Rebalance 1 Hari)", linewidth=2, color='green')
    plt.title("Rolling HRP Portfolio Performance (Rebalance 1 Hari)")
    plt.ylabel("Portfolio Value")
    plt.xlabel("Date")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # =======================================
    # 5. DRAWDOWN VISUALIZATION - DIPERBAIKI
    # =======================================
    running_max = final_portfolio.cummax()
    drawdown = (final_portfolio - running_max) / running_max

    plt.figure(figsize=(12,4))
    plt.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)
    plt.plot(drawdown, color='red', label='Drawdown', linewidth=1.5)
    plt.title("Drawdown Rolling HRP Portfolio (Rebalance 1 Hari)")
    plt.ylabel("Drawdown")
    plt.xlabel("Date")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # =======================================
    # 6. CETAK BOBOT WINDOW TERAKHIR
    # =======================================
    print("\n" + "="*60)
    print("HRP PORTFOLIO RESULTS")
    print("="*60)
    
    if weights_all:
        print("\nBobot HRP pada window terakhir:")
        last_weights = weights_all[-1].sort_values(ascending=False)
        for asset, weight in last_weights.items():
            print(f"  {asset}: {weight:.3%}")

    # =======================================
    # 7. METRIK PORTOFOLIO LENGKAP - DIPERBAIKI
    # =======================================
    
    # --- Return portofolio sebagai Series untuk perhitungan ---
    port_returns_series = pd.Series(portfolio_returns_list, 
                                   index=returns.index[window_train:window_train + len(portfolio_returns_list)])
    
    # --- Turnover ---
    turnovers = []
    for i in range(1, len(weights_all)):
        turnover = np.sum(np.abs(weights_all[i] - weights_all[i-1]))
        turnovers.append(turnover)
    avg_turnover = np.mean(turnovers) if turnovers else 0

    # --- Sum of Squared Portfolio Weights (SSPW) ---
    sspw = np.mean([np.sum(w ** 2) for w in weights_all]) if weights_all else 0

    # --- Sharpe Ratio ---
    rf = 0.000110177  # risk-free rate 4%/tahun
    if len(port_returns_series) > 1:
        sharpe_daily = (port_returns_series.mean() - rf) / port_returns_series.std()
    else:
        sharpe_daily = 0

    # --- Adjusted Sharpe Ratio ---
    if len(port_returns_series) > 1:
        S = skew(port_returns_series)
        K = kurtosis(port_returns_series, fisher=True)
        asr_daily = sharpe_daily * (1 + ((S / 6) * sharpe_daily) - ((K - 3) / 24) * sharpe_daily**2)
    else:
        asr_daily = 0

    # --- Certainty Equivalent Return (CER) ---
    gamma = 1  # koefisien aversi risiko
    if len(port_returns_series) > 1:
        mean_r = port_returns_series.mean()
        var_r = port_returns_series.var()
        cer_daily = (mean_r-rf) - (0.5 * gamma * var_r)
    else:
        cer_daily = 0

    # --- Maximum Drawdown ---
    max_drawdown = drawdown.min()

    # --- Total Return ---
    total_return = final_portfolio.iloc[-1] - 1 if len(final_portfolio) > 0 else 0

    # --- Cetak semua metrik ---
    print(f"\nPERFORMANCE METRICS:")
    print(f"Total Return: {total_return:.2%}")
    print(f"Maximum Drawdown: {max_drawdown:.2%}")
    print(f"Average Turnover: {avg_turnover:.4f}")
    print(f"SSPW: {sspw:.4f}")
    print(f"Sharpe Ratio (daily): {sharpe_daily:.4f}")
    print(f"Adjusted Sharpe Ratio (daily): {asr_daily:.4f}")
    print(f"Certainty Equivalent Return (daily, gamma=1): {cer_daily:.6f}")
    print(f"Number of Rolling Windows: {len(weights_all)}")
    print(f"Number of Trading Days: {len(portfolio_values)}")
    print(f"Rebalancing Frequency: Every {step} days")

    # =======================================
    # 8. VISUALISASI TURNOVER - DITAMBAHKAN
    # =======================================
    plt.figure(figsize=(12,4))
    plt.plot(turnovers, color='blue', alpha=0.7, marker='o', markersize=3)
    plt.axhline(y=avg_turnover, color='red', linestyle='--', 
                label=f'Average Turnover: {avg_turnover:.4f}')
    plt.title("Portfolio Turnover Over Time (HRP)")
    plt.ylabel("Turnover")
    plt.xlabel("Rolling Window")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
# =======================================
# DETAILED ANALYSIS FOR FIRST WINDOW ONLY - DITAMBAHKAN DI AKHIR
# =======================================
print("\n" + "="*80)
print("DETAILED ANALYSIS - WINDOW PERTAMA")
print("="*80)

# Ambil data window pertama dari weights_all jika tersedia
if weights_all:
    # Recreate the first window processing
    t = 0
    window_train = 254
    train_first = returns.iloc[t:t+window_train]
    
    print(f"Window pertama: dari hari {t} sampai {t+window_train-1}")
    print(f"Jumlah aset: {len(train_first.columns)}")
    print(f"Periode training: {len(train_first)} hari")
    
    # --- 1. Membentuk Matriks Korelasi ---
    print("\n1. MEMBENTUK MATRIKS KORELASI")
    print("-" * 40)
    corr_matrix_first = train_first.corr()
    cov_matrix_first = train_first.cov()
    
    
    print("Dimensi matriks korelasi:", corr_matrix_first.shape)
    print("\n5 nilai korelasi tertinggi:")
    corr_vals = []
    for i in range(len(corr_matrix_first.columns)):
        for j in range(i+1, len(corr_matrix_first.columns)):
            corr_vals.append((corr_matrix_first.iloc[i,j], 
                             corr_matrix_first.columns[i], 
                             corr_matrix_first.columns[j]))
    
    corr_vals.sort(reverse=True)
    for i, (corr_val, asset1, asset2) in enumerate(corr_vals[:5]):
        print(f"  {asset1} - {asset2}: {corr_val:.4f}")
    
    print("\n5 nilai korelasi terendah:")
    for i, (corr_val, asset1, asset2) in enumerate(corr_vals[-5:]):
        print(f"  {asset1} - {asset2}: {corr_val:.4f}")
    
    # Visualisasi matriks korelasi
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix_first, dtype=bool))
    sns.heatmap(corr_matrix_first, mask=mask, cmap='coolwarm', center=0, annot=True)
    plt.title('Matriks Korelasi - Window Pertama')
    plt.tight_layout()
    plt.show()
    
    # --- 2. Klasterisasi Hierarki ---
    print("\n2. KLASTERISASI HIERARKI")
    print("-" * 40)
    dist_first = correl_dist(corr_matrix_first)
    link_first = linkage(dist_first, method='single')
    ordered_idx_first = leaves_list(link_first)
    ordered_tickers_first = list(train_first.columns[ordered_idx_first])
    
    print("Urutan aset setelah klasterisasi:")
    for i, ticker in enumerate(ordered_tickers_first):
        print(f"  {i+1:2d}. {ticker}")
    
    # Visualisasi dendrogram
    plt.figure(figsize=(15, 8))
    dendrogram(link_first, labels=train_first.columns, leaf_rotation=90, color_threshold=0)
    plt.title('Dendrogram - Klasterisasi Hierarki (Window Pertama)')
    plt.xlabel('Aset Kripto')
    plt.ylabel('Distance')
    plt.tight_layout()
    plt.show()
    
    # --- 3. Quasi Diagonalization ---
    print("\n3. QUASI DIAGONALIZATION")
    print("-" * 40)
    
    # Reorder matriks korelasi berdasarkan hasil klasterisasi
    corr_reordered = corr_matrix_first.loc[ordered_tickers_first, ordered_tickers_first]
    mask = np.triu(np.ones_like(corr_reordered, dtype=bool))

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_reordered, mask=mask, cmap='coolwarm', center=0, annot=True)
    plt.title('Matriks Korelasi Setelah Quasi Diagonalization')
    plt.tight_layout()
    plt.show()
    
    # --- 4. Pembelahan Recursive Detail ---
    print("\n4. PEMBELAHAN RECURSIVE DETAIL")
    print("-" * 40)
    
    cov_df_first = pd.DataFrame(cov_matrix_first, index=train_first.columns, columns=train_first.columns)
    cov_df_first_hrp= cov_df_first.loc[ordered_tickers_first, ordered_tickers_first]
    #cov_df_first_hrp.to_excel("/Users/galvanfattah/Downloads/cov_matrix_first_hrp.xlsx")
    #cov_df_first.to_excel("/Users/galvanfattah/Downloads/cov_matrix_first.xlsx")

    def detailed_recursive_bisection(cov_df, ordered_tickers):
        weights = pd.Series(1.0, index=ordered_tickers)
        clusters = [ordered_tickers]
        cluster_level = 0
        
        while clusters:
            new_clusters = []
            print(f"\nLevel {cluster_level}: {len(clusters)} cluster")
            
            for i, cluster in enumerate(clusters):
                if len(cluster) <= 1:
                    print(f"  Cluster {i+1}: {cluster} (tidak dibelah)")
                    continue
                    
                split = len(cluster) // 2
                cluster1 = cluster[:split]
                cluster2 = cluster[split:]
                
                var1 = get_cluster_var(cov_df, cluster1)
                var2 = get_cluster_var(cov_df, cluster2)
                alpha = 1 - var1 / (var1 + var2)
                
                print(f"  Cluster {i+1}: {len(cluster)} aset")
                print(f"    Cluster 1: {cluster1}")
                print(f"    Cluster 2: {cluster2}")
                print(f"    Variance cluster 1: {var1:.6f}")
                print(f"    Variance cluster 2: {var2:.6f}")
                print(f"    Alpha: {alpha:.4f}")
                
                for asset in cluster1:
                    weights[asset] *= alpha
                for asset in cluster2:
                    weights[asset] *= (1 - alpha)
                    
                new_clusters.extend([cluster1, cluster2])
                
            clusters = new_clusters
            cluster_level += 1
            
        return weights
    
    weights_first_detailed = detailed_recursive_bisection(cov_df_first, ordered_tickers_first)
    weights_first_detailed /= weights_first_detailed.sum()
    
    # --- 5. Bobot yang Didapat ---
    print("\n5. BOBOT PORTOFOLIO AKHIR - WINDOW PERTAMA")
    print("-" * 40)
    
    # Urutkan bobot dari terbesar ke terkecil
    sorted_weights = weights_first_detailed.sort_values(ascending=False)
    
    print("\nBobot per aset (diurutkan):")
    for asset, weight in sorted_weights.items():
        print(f"  {asset}: {weight:.3%}")
    
    print(f"\nStatistik bobot:")
    print(f"  Rata-rata: {weights_first_detailed.mean():.3%}")
    print(f"  Standar deviasi: {weights_first_detailed.std():.3%}")
    print(f"  Minimum: {weights_first_detailed.min():.3%}")
    print(f"  Maximum: {weights_first_detailed.max():.3%}")
        # --- 6. PERHITUNGAN RETURN 1 HARI PADA DATA TESTING ---
    print("\n6. PERHITUNGAN RETURN 1 HARI PADA DATA TESTING")
    print("-" * 40)
    
    # Definisikan test_first untuk window pertama
    test_first = returns.iloc[t+window_train:t+window_train+1]
    print(test_first)
    if not test_first.empty:
        test_date = test_first.index[0]
        test_returns = test_first.iloc[0]
        port_return = test_returns @ weights_first_detailed
        
        print(f"Tanggal testing: {test_date}")
        print(f"Return portofolio: {port_return:.4%}")
            
    else:
        print("Tidak ada data testing untuk window pertama")
else:
    print("Tidak ada data weights_all untuk dianalisis")

# =======================================
# 9. OUTPUT SEMUA BOBOT HRP TIAP REBALANCING
# =======================================

# Tanggal rebalancing = awal periode testing setiap window
rebalance_dates = returns.index[window_train:window_train + len(weights_all)]

# Gabungkan semua bobot ke DataFrame
weights_all_df = pd.DataFrame(weights_all, index=rebalance_dates)

# Urutkan kolom aset agar konsisten
weights_all_df = weights_all_df.sort_index(axis=1)

print("\nSEMUA BOBOT HRP PER REBALANCING WINDOW:")
print(weights_all_df)

weights_all_df.to_excel("/Users/galvanfattah/Downloadshrp_weights_all_windows.xlsx")

# (opsional) simpan ke file
# weights_all_df.to_csv("hrp_weights_all_windows.csv")

# HCAA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from scipy.stats import skew, kurtosis
import seaborn as sns

# =======================================
# 1. BACA DATA
# =======================================
# prices = pd.read_csv('crypto_prices.csv', index_col=0, parse_dates=True)
returns = prices.pct_change().dropna()

print("Data return shape:", returns.shape)
print(returns.head())

# =======================================
# 2. FUNGSI DASAR & GAP STATISTIC
# =======================================

def correl_dist(corr):
    """Konversi korelasi ke jarak"""
    return np.sqrt(0.5 * (1 - corr))

def compute_Wk(dist_matrix, labels):
    """Within-cluster dispersion"""
    if isinstance(dist_matrix, pd.DataFrame):
        dist_matrix = dist_matrix.values
    
    clusters = np.unique(labels)
    Wk = 0
    for c in clusters:
        idx = np.where(labels == c)[0]
        if len(idx) > 1:
            sub_dist = dist_matrix[np.ix_(idx, idx)]
            Wk += np.sum(sub_dist) / (2 * len(idx))
    return Wk

def gap_statistic_corrected(returns_data, max_clusters=20, B=100, method='ward', random_seed=42):
    """Gap Statistic untuk menentukan jumlah cluster optimal"""
    np.random.seed(random_seed)
    
    n_periods, n_assets = returns_data.shape
    max_clusters = min(max_clusters, n_assets - 1)
    
    if max_clusters < 2:
        return 2
    
    # 1. Hitung untuk data asli
    corr_matrix = returns_data.corr()
    dist_matrix = correl_dist(corr_matrix)
    dist_condensed = squareform(dist_matrix, checks=False)
    
    log_Wks = []

    for k in range(1, max_clusters + 1):
        link = linkage(dist_condensed, method=method)
        labels = fcluster(link, k, criterion='maxclust')
        Wk = compute_Wk(dist_matrix, labels)
        log_Wks.append(np.log(Wk) if Wk > 0 else 0.0)

    # 2. Hitung untuk data referensi
    log_Wkbs_bootstrap = []

    for b in range(B):
        ref_returns = np.random.uniform(0, 1, size=(n_periods, n_assets))
        ref_returns_df = pd.DataFrame(ref_returns, columns=returns_data.columns)
        
        ref_corr = ref_returns_df.corr()
        ref_dist = correl_dist(ref_corr)
        ref_condensed = squareform(ref_dist, checks=False)
        
        log_Wkbs = []
        
        for k in range(1, max_clusters + 1):
            link_ref = linkage(ref_condensed, method=method)
            labels_ref = fcluster(link_ref, k, criterion='maxclust')
            
            Wkb = compute_Wk(ref_dist, labels_ref)
            log_Wkb = np.log(Wkb) if Wkb > 0 else 0.0
            log_Wkbs.append(log_Wkb)
        
        log_Wkbs_bootstrap.append(log_Wkbs)

    # 3. Hitung statistik gap
    log_Wkbs_bootstrap = np.array(log_Wkbs_bootstrap)
    E_logWkb = np.mean(log_Wkbs_bootstrap, axis=0)
    gaps = E_logWkb - np.array(log_Wks)
    
    # 4. Hitung standar deviasi
    sd_k = np.std(log_Wkbs_bootstrap, axis=0, ddof=1)
    s_k = sd_k * np.sqrt(1 + 1/B)
    
    # 5. Pilih jumlah cluster optimal
    optimal_k = 1
    for k in range(len(gaps)-1):
        if k+1 < len(s_k) and gaps[k] >= gaps[k+1] - s_k[k+1]:
            optimal_k = k + 1
            break
    else:
        if len(gaps) > 0:
            optimal_k = np.argmax(gaps) + 1
        else:
            optimal_k = 1
    
    optimal_k = max(1, min(optimal_k, max_clusters))
    
    return optimal_k

# =======================================
# 3. IMPLEMENTASI HCAA YANG BENAR - REKURSIF 50-50%
# =======================================

def get_subtree_asset_indices(linkage_matrix, node_idx, n_assets):
    if node_idx < n_assets:
        return [int(node_idx)]
    row = int(node_idx - n_assets)
    left = int(linkage_matrix[row, 0])
    right = int(linkage_matrix[row, 1])
    left_idx = get_subtree_asset_indices(linkage_matrix, left, n_assets)
    right_idx = get_subtree_asset_indices(linkage_matrix, right, n_assets)
    return left_idx + right_idx

def top_down_assign_weights(linkage_matrix, node_idx, assign_weight, cluster_labels, n_assets, weights_array):
    leaf_indices = get_subtree_asset_indices(linkage_matrix, node_idx, n_assets)
    # check cluster labels among these leaves
    labels_set = set([cluster_labels[i] for i in leaf_indices])
    if len(labels_set) == 1:
        # STOP: assign equal weight to all leaves here
        per_asset = assign_weight / len(leaf_indices)
        for li in leaf_indices:
            weights_array[li] += per_asset
        return
    # else, split into children
    # if node is leaf (shouldn't happen because labels_set>1), handle defensively
    if node_idx < n_assets:
        weights_array[node_idx] += assign_weight
        return
    row = int(node_idx - n_assets)
    left = int(linkage_matrix[row, 0])
    right = int(linkage_matrix[row, 1])
    # split 50-50
    half = assign_weight * 0.5
    top_down_assign_weights(linkage_matrix, left, half, cluster_labels, n_assets, weights_array)
    top_down_assign_weights(linkage_matrix, right, half, cluster_labels, n_assets, weights_array)

def hcaa_weights_from_linkage(returns, n_clusters, linkage_method='ward'):
    corr = returns.corr()
    dist = correl_dist(corr)
    dist_condensed = squareform(dist, checks=False)
    link = linkage(dist_condensed, method=linkage_method)
    n_assets = len(returns.columns)
    asset_names = returns.columns.tolist()

    # cluster labels from cut
    cluster_labels = fcluster(link, n_clusters, criterion='maxclust')  # labels 1..k

    # prepare weights array
    weights_array = np.zeros(n_assets)

    # find root node index (last merge)
    last_row = link.shape[0] - 1  # index in linkage
    root_node = n_assets + last_row

    # start recursion: full weight 1.0 at root
    top_down_assign_weights(link, root_node, 1.0, cluster_labels, n_assets, weights_array)

    # create Series aligned to asset names and normalize
    weights_series = pd.Series(weights_array, index=asset_names)
    weights_series = weights_series / weights_series.sum()

    # also return link and clusters list for visualization
    clusters = {}
    for i, a in enumerate(asset_names):
        clusters.setdefault(cluster_labels[i], []).append(a)

    return weights_series, link, list(clusters.values())

# =======================================
# 4. FUNGSI ROLLING WINDOW HCAA
# =======================================

def precompute_optimal_clusters_all_linkages(returns, window_train=254, step=1, max_clusters=20):
    linkage_methods = ['ward', 'complete', 'average', 'single']
    optimal_clusters_dict = {}
    
    n_rolls = len(returns) - window_train
    
    for linkage_method in linkage_methods:
        cluster_counts = []
        print(f"  Computing for {linkage_method} linkage...")
        
        for t in range(0, n_rolls, step):
            if t + window_train >= len(returns):
                break
                
            train_data = returns.iloc[t:t + window_train]
            
            try:
                n_clusters = gap_statistic_corrected(
                    train_data,
                    max_clusters=min(max_clusters, len(train_data.columns)-1),
                    method=linkage_method,
                    random_seed=42,
                    B=50
                )
                n_clusters = max(1, n_clusters)
                cluster_counts.append(n_clusters)
                
            except Exception as e:
                cluster_counts.append(2)
                continue
        
        optimal_clusters_dict[linkage_method] = cluster_counts
        print(f"    {linkage_method}: {len(cluster_counts)} windows computed")
    
    return optimal_clusters_dict

def run_rolling_hcaa_consistent(returns, linkage_method='ward', step=1, precomputed_clusters=None):    
    window_train = 254
    n_rolls = len(returns) - window_train
    
    portfolio_values = [1.0]
    weights_all = []
    portfolio_returns = []
    cluster_counts = []

    print(f"Running HCAA: {linkage_method.upper()} linkage")
    
    for t in range(0, n_rolls, step):
        if t + window_train + 1 >= len(returns):
            break
            
        train_data = returns.iloc[t:t + window_train]
        test_data = returns.iloc[t + window_train:t + window_train + 1]
        
        if test_data.empty or len(train_data) < window_train:
            continue

        try:
            # 1. Gunakan PRE-COMPUTED clusters
            if precomputed_clusters and linkage_method in precomputed_clusters:
                window_idx = t // step
                if window_idx < len(precomputed_clusters[linkage_method]):
                    n_clusters = precomputed_clusters[linkage_method][window_idx]
                else:
                    n_clusters = 2
            else:
                n_clusters = gap_statistic_corrected(
                    train_data, 
                    max_clusters=min(8, len(train_data.columns)-1),
                    method=linkage_method,
                    random_seed=42,
                    B=50
                )
                n_clusters = max(2, n_clusters)
            
            cluster_counts.append(n_clusters)

            # 2. Dapatkan weights dengan HCAA YANG BENAR
            weights, linkage_matrix, clusters = hcaa_weights_from_linkage(
                train_data, 
                n_clusters=n_clusters,
                linkage_method=linkage_method
            )

            # Normalisasi weights
            weights = weights / weights.sum()
            weights_all.append(weights)

            # 3. Hitung portfolio return
            port_return = (test_data.values.flatten() @ weights.values).item()
            portfolio_returns.append(port_return)
            
            # 4. Update portfolio value
            new_value = portfolio_values[-1] * (1 + port_return)
            portfolio_values.append(new_value)
            
        except Exception as e:
            print(f"Error in window {t}: {e}")
            n_assets = len(train_data.columns)
            weights = pd.Series(1.0/n_assets, index=train_data.columns)
            weights_all.append(weights)
            cluster_counts.append(2)
            port_return = test_data.mean().mean()
            portfolio_returns.append(port_return)
            new_value = portfolio_values[-1] * (1 + port_return)
            portfolio_values.append(new_value)
            continue

    return {
        'portfolio_values': portfolio_values,
        'weights_all': weights_all,
        'portfolio_returns': portfolio_returns,
        'cluster_counts': cluster_counts
    }

# =======================================
# 5. FUNGSI CALCULATE METRICS
# =======================================

def calculate_performance_metrics(results, returns, linkage_method):
    portfolio_values = results['portfolio_values']
    weights_all = results['weights_all']
    portfolio_returns = results['portfolio_returns']
    
    if len(portfolio_values) <= 1:
        return None
    
    final_portfolio = pd.Series(portfolio_values[1:], 
                               index=returns.index[254:254 + len(portfolio_values[1:])])
    
    port_returns_series = pd.Series(portfolio_returns, 
                                   index=returns.index[254:254 + len(portfolio_returns)])
    
    # Turnover
    turnovers = []
    for i in range(1, len(weights_all)):
        turnover = np.sum(np.abs(weights_all[i] - weights_all[i-1]))
        turnovers.append(turnover)
    avg_turnover = np.mean(turnovers) if turnovers else 0

    # Sum of Squared Portfolio Weights (SSPW)
    sspw = np.mean([np.sum(w ** 2) for w in weights_all]) if weights_all else 0

    # Sharpe Ratio
    rf = 0.000110177
    if len(port_returns_series) > 1:
        sharpe_ratio = (port_returns_series.mean() - rf) / port_returns_series.std()
    else:
        sharpe_ratio = 0

    # Adjusted Sharpe Ratio
    if len(port_returns_series) > 1:
        S = skew(port_returns_series)
        K = kurtosis(port_returns_series, fisher=False)
        adjusted_sharpe = sharpe_ratio * (1 + (S / 6) * sharpe_ratio - ((K - 3) / 24) * sharpe_ratio**2)
    else:
        adjusted_sharpe = 0

    # Certainty Equivalent Return (CER)
    gamma = 1
    if len(port_returns_series) > 1:
        mean_return = port_returns_series.mean()
        variance_return = port_returns_series.var()
        cer = (mean_return-rf) - (0.5 * gamma * variance_return)
    else:
        cer = 0

    # Maximum Drawdown
    running_max = final_portfolio.cummax()
    drawdown = (final_portfolio - running_max) / running_max
    max_drawdown = drawdown.min()

    # Total Return
    total_return = final_portfolio.iloc[-1] - 1 if len(final_portfolio) > 0 else 0

    return {
        'final_portfolio': final_portfolio,
        'port_returns_series': port_returns_series,
        'drawdown': drawdown,
        'turnovers': turnovers,
        'total_return': total_return,
        'max_drawdown': max_drawdown,
        'avg_turnover': avg_turnover,
        'sspw': sspw,
        'sharpe_ratio': sharpe_ratio,
        'adjusted_sharpe': adjusted_sharpe,
        'cer': cer,
        'linkage_method': linkage_method
    }

# =======================================
# 6. MAIN EXECUTION
# =======================================

# Definisikan semua kombinasi
linkage_methods = ['ward', 'complete', 'average', 'single']

print("=" * 80)
print("HCAA ANALYSIS - TRUE RECURSIVE 50-50% DIVISION")
print("=" * 80)

# STEP 1: PRE-COMPUTE optimal clusters
precomputed_clusters = precompute_optimal_clusters_all_linkages(returns, window_train=254, step=1, max_clusters=20)

# STEP 2: Jalankan HCAA untuk semua kombinasi
all_results = {}
all_clusters_data = {}

for linkage_method in linkage_methods:
    combo_name = f"{linkage_method}_linkage"
    
    print(f"\nProcessing: {combo_name.upper()}")
    
    try:
        rolling_results = run_rolling_hcaa_consistent(
            returns,
            linkage_method=linkage_method,
            precomputed_clusters=precomputed_clusters
        )
        
        metrics = calculate_performance_metrics(rolling_results, returns, linkage_method)
        
        if metrics:
            all_results[combo_name] = metrics
            all_clusters_data[combo_name] = rolling_results['cluster_counts']
            print(f"✓ SUCCESS: {combo_name}")
            print(f"  Final Value: {metrics['final_portfolio'].iloc[-1]:.4f}")
            print(f"  Total Return: {metrics['total_return']:.2%}")
            print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.4f}")
            print(f"  Avg Clusters: {np.mean(rolling_results['cluster_counts']):.1f}")
        else:
            print(f"✗ SKIPPED: {combo_name} - insufficient data")
            
    except Exception as e:
        print(f"✗ ERROR: {combo_name} - {str(e)}")
        continue

print(f"\nCompleted {len(all_results)} out of {len(linkage_methods)} combinations")

# =======================================
# 7. VISUALISASI DAN ANALISIS LENGKAP
# =======================================

if all_results:
    # Plot 1: Portfolio Performance Comparison
    plt.figure(figsize=(16, 10))
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_results)))
    
    for i, (combo_name, metrics) in enumerate(all_results.items()):
        label = f"{metrics['linkage_method'].upper()}-L"
        plt.plot(metrics['final_portfolio'].index, metrics['final_portfolio'].values, 
                label=label, linewidth=2, color=colors[i])
    
    plt.title("HCAA Portfolio Performance - True Recursive 50-50% Division", fontsize=14)
    plt.ylabel("Portfolio Value")
    plt.xlabel("Date")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Plot 2: Drawdown Comparison
    plt.figure(figsize=(16, 8))
    
    for i, (combo_name, metrics) in enumerate(all_results.items()):
        label = f"{metrics['linkage_method'].upper()}-L"
        plt.plot(metrics['drawdown'].index, metrics['drawdown'].values * 100, 
                label=label, linewidth=1.5, color=colors[i])
    
    plt.title("Drawdown Comparison - All Linkage Methods", fontsize=14)
    plt.ylabel("Drawdown (%)")
    plt.xlabel("Date")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Plot 3: Turnover Over Time
    plt.figure(figsize=(16, 8))
    
    for i, (combo_name, metrics) in enumerate(all_results.items()):
        label = f"{metrics['linkage_method'].upper()}-L"
        plt.plot(metrics['turnovers'], label=label, linewidth=1.5, color=colors[i], marker='o', markersize=3)
    
    plt.title("Portfolio Turnover Over Time - All Linkage Methods", fontsize=14)
    plt.ylabel("Turnover")
    plt.xlabel("Rolling Window")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Plot 4: Cluster Evolution
    plt.figure(figsize=(16, 8))
    
    for i, (combo_name, clusters) in enumerate(all_clusters_data.items()):
        if clusters:
            label = f"{all_results[combo_name]['linkage_method'].upper()}-L"
            plt.plot(range(len(clusters)), clusters, label=label, linewidth=2, 
                    color=colors[i], marker='s', markersize=4, alpha=0.8)
    
    plt.title("Cluster Evolution Over Time - All Linkage Methods", fontsize=14)
    plt.ylabel("Number of Clusters")
    plt.xlabel("Rolling Window")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # =======================================
    # 8. TABEL PERBANDINGAN KOMPREHENSIF
    # =======================================
    print("\n" + "="*120)
    print("COMPREHENSIVE COMPARISON: HCAA WITH DIFFERENT LINKAGE METHODS")
    print("="*120)
    
    comparison_data = []
    for combo_name, metrics in all_results.items():
        # Calculate additional metrics
        volatility_annual = metrics['port_returns_series'].std() * np.sqrt(252)
        var_95 = metrics['port_returns_series'].quantile(0.05)
        cvar_95 = metrics['port_returns_series'][metrics['port_returns_series'] <= var_95].mean()
        
        comparison_data.append({
            'Linkage': metrics['linkage_method'].upper(),
            'Total_Return': f"{metrics['total_return']:.2%}",
            'Max_Drawdown': f"{metrics['max_drawdown']:.2%}",
            'Volatility_Ann': f"{volatility_annual:.2%}",
            'Sharpe_Ratio': f"{metrics['sharpe_ratio']:.4f}",
            'Adj_Sharpe': f"{metrics['adjusted_sharpe']:.4f}",
            'CER': f"{metrics['cer']:.6f}",
            'Avg_Turnover': f"{metrics['avg_turnover']:.4f}",
            'SSPW': f"{metrics['sspw']:.4f}",
            'VaR_95%': f"{var_95:.4f}",
            'CVaR_95%': f"{cvar_95:.4f}"
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df['Sharpe_Value'] = comparison_df['Sharpe_Ratio'].str.replace('%', '').astype(float)
    comparison_df = comparison_df.sort_values('Sharpe_Value', ascending=False)
    comparison_df = comparison_df.drop('Sharpe_Value', axis=1)
    
    print(comparison_df.to_string(index=False))

    # =======================================
    # 9. IDENTIFIKASI PERFORMER TERBAIK
    # =======================================
    best_combo_sharpe = max(all_results.keys(), key=lambda x: all_results[x]['sharpe_ratio'])
    best_combo_cer = max(all_results.keys(), key=lambda x: all_results[x]['cer'])
    best_combo_turnover = min(all_results.keys(), key=lambda x: all_results[x]['avg_turnover'])
    best_combo_drawdown = min(all_results.keys(), key=lambda x: all_results[x]['max_drawdown'])
    
    best_metrics_sharpe = all_results[best_combo_sharpe]
    best_metrics_cer = all_results[best_combo_cer]
    best_metrics_turnover = all_results[best_combo_turnover]
    best_metrics_drawdown = all_results[best_combo_drawdown]
    
    print(f"\n{'='*80}")
    print(f"BEST PERFORMING COMBINATION BY SHARPE RATIO: {best_combo_sharpe.upper()}")
    print(f"{'='*80}")
    
    print(f"Linkage Method: {best_metrics_sharpe['linkage_method'].upper()}")
    print(f"Total Return: {best_metrics_sharpe['total_return']:.2%}")
    print(f"Maximum Drawdown: {best_metrics_sharpe['max_drawdown']:.2%}")
    print(f"Sharpe Ratio: {best_metrics_sharpe['sharpe_ratio']:.4f}")
    print(f"Adjusted Sharpe Ratio: {best_metrics_sharpe['adjusted_sharpe']:.4f}")
    print(f"Certainty Equivalent Return (CER): {best_metrics_sharpe['cer']:.6f}")
    print(f"Average Turnover: {best_metrics_sharpe['avg_turnover']:.4f}")
    print(f"SSPW: {best_metrics_sharpe['sspw']:.4f}")

    print(f"\n{'='*80}")
    print(f"BEST PERFORMING COMBINATION BY LOWEST TURNOVER: {best_combo_turnover.upper()}")
    print(f"{'='*80}")
    
    print(f"Linkage Method: {best_metrics_turnover['linkage_method'].upper()}")
    print(f"Total Return: {best_metrics_turnover['total_return']:.2%}")
    print(f"Maximum Drawdown: {best_metrics_turnover['max_drawdown']:.2%}")
    print(f"Sharpe Ratio: {best_metrics_turnover['sharpe_ratio']:.4f}")
    print(f"Average Turnover: {best_metrics_turnover['avg_turnover']:.4f}")

else:
    print("No successful results obtained!")

print("\n🎯 TRUE HCAA ANALYSIS COMPLETED!")

# =======================================
# ANALISIS SINGKAT WINDOW PERTAMA - HCAA
# =======================================

def analisis_singkat_window_pertama(returns, linkage_method='single'):
    """Analisis singkat window pertama HCAA - fokus 3 poin utama"""
    print(f"\nANALISIS SINGKAT WINDOW PERTAMA - {linkage_method.upper()} LINKAGE")
    print("=" * 60)
    
    # 1. DATA PREPARATION
    window_train = 254
    train = returns.iloc[0:window_train]
    test = returns.iloc[window_train:window_train + 1]
    
    print(f"1. DATA: {len(train.columns)} aset, {len(train)} periode")
    
    # 2. MATRIKS KORELASI
    print("\n2. MATRIKS KORELASI")
    corr_matrix = train.corr()
    corr_values = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]
    print(f"   Korelasi: rata-rata={corr_values.mean():.4f}, min={corr_values.min():.4f}, max={corr_values.max():.4f}")
    
    # Visualisasi korelasi
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0, xticklabels=True, yticklabels=True)
    plt.title('Matriks Korelasi')
    plt.tight_layout()
    plt.show()
    
    # 3. STRUKTUR HIERARKI - SINGLE LINKAGE
    print(f"\n3. STRUKTUR HIERARKI - {linkage_method.upper()} LINKAGE")
    dist_matrix = correl_dist(corr_matrix)
    dist_condensed = squareform(dist_matrix, checks=False)
    link = linkage(dist_condensed, method=linkage_method)
    
    # Dendrogram
    plt.figure(figsize=(12, 6))
    dendrogram(link, labels=train.columns.tolist(), leaf_rotation=90, color_threshold=0)
    plt.title(f'Struktur Hierarki - {linkage_method.capitalize()} Linkage')
    plt.show()
    
    # 4. JUMLAH CLUSTER OPTIMAL
    optimal_k = gap_statistic_corrected(train, max_clusters=min(10, len(train.columns)-1), 
                                      method=linkage_method, B=30)
    print(f"   Jumlah cluster optimal: {optimal_k}")
    
    # 5. PROSES PEMBOBOTAN HCAA - DETAIL PER ASET
    print("\n4. PROSES PEMBOBOTAN HCAA REKURSIF 50-50%")
    print("-" * 50)
    
    # Dapatkan weights dengan HCAA YANG BENAR
    final_weights, link_matrix, final_clusters = hcaa_weights_from_linkage(
        train, n_clusters=optimal_k, linkage_method=linkage_method
    )
    
    # Tampilkan detail pembobotan
    cluster_labels = fcluster(link, optimal_k, criterion='maxclust')
    cluster_df = pd.DataFrame({'asset': train.columns, 'cluster': cluster_labels, 'weight': final_weights})
    
    print(f"\nDISTRIBUSI CLUSTER (k={optimal_k}):")
    for cluster_id in sorted(cluster_df['cluster'].unique()):
        cluster_assets = cluster_df[cluster_df['cluster'] == cluster_id]
        total_weight = cluster_assets['weight'].sum()
        print(f"Cluster {cluster_id}: {len(cluster_assets)} aset, total weight: {total_weight:.4f}")
    
    print(f"\nDETAIL BOBOT PER ASET:")
    print(f"{'Asset':<15} {'Cluster':<8} {'Weight':<10} {'%':<8}")
    print("-" * 45)
    for _, row in cluster_df.iterrows():
        print(f"{row['asset']:<15} {row['cluster']:<8} {row['weight']:<10.4f} {row['weight']*100:<7.2f}%")
    
    # 6. PERFORMANCE
    print("\n5. PERFORMANCE TEST")
    port_ret = (test.values.flatten() @ final_weights.values).item()
    equal_weights = np.ones(len(train.columns)) / len(train.columns)
    equal_ret = (test.values.flatten() @ equal_weights).item()
    
    print(f"Return HCAA: {port_ret:.4%}")
    print(f"Return Equal Weight: {equal_ret:.4%}")
    print(f"Excess Return: {port_ret - equal_ret:+.4%}")
    
    # 7. STATISTIK BOBOT
    print("\n6. STATISTIK BOBOT")
    herfindahl = np.sum(final_weights ** 2)
    effective_n = 1 / herfindahl
    print(f"Effective Number of Assets: {effective_n:.2f}")
    print(f"Herfindahl Index: {herfindahl:.4f}")
    print(f"Bobot min: {final_weights.min():.4f}, max: {final_weights.max():.4f}")
    
    return {
        'weights': final_weights,
        'cluster_df': cluster_df,
        'optimal_k': optimal_k,
        'return': port_ret,
        'excess_return': port_ret - equal_ret
    }

# JALANKAN ANALISIS SINGKAT
print("\n" + "="*80)
print("ANALISIS SINGKAT WINDOW PERTAMA")
print("="*80)

# Analisis untuk single linkage (bisa ganti dengan metode lain)
result_single = analisis_singkat_window_pertama(returns, linkage_method='single')

print("\n" + "="*80)
print("ANALISIS SINGKAT SELESAI")
print("="*80)

#=======================================
# 11. ANALISIS MENDALAM GAP STATISTIC WINDOW PERTAMA
# =======================================

def analisis_gap_statistic_mendalam():
    """Analisis mendalam perhitungan Gap Statistic pada window pertama"""
    print("\n" + "="*80)
    print("ANALISIS MENDALAM GAP STATISTIC - WINDOW PERTAMA")
    print("="*80)
    
    # Ambil data window pertama
    window_train = 254
    train = returns.iloc[0:window_train]
    
    print("DATA PREPARATION")
    print(f"Jumlah aset: {len(train.columns)}")
    print(f"Jumlah periode: {len(train)}")
    
    # 1. HITUNG MATRIKS KORELASI & JARAK
    print("\n" + "-"*50)
    print("1. PERHITUNGAN MATRIKS KORELASI DAN JARAK")
    print("-"*50)
    
    corr_matrix = train.corr()
    dist_matrix = correl_dist(corr_matrix)
    dist_condensed = squareform(dist_matrix, checks=False)
    
    corr_values = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]
    print(f"Korelasi rata-rata: {corr_values.mean():.4f}")
    print(f"Korelasi min-maks: [{corr_values.min():.4f}, {corr_values.max():.4f}]")
    
    # 2. PERHITUNGAN DISPERSI DATA ASLI (Wk)
    print("\n" + "-"*50)
    print("2. PERHITUNGAN DISPERSI DATA ASLI (Wk)")
    print("-"*50)
    
    max_clusters = min(20, len(train.columns) - 1)
    Wks_original = []
    log_Wks = []
    
    print(f"\n{'k':<3} {'Wk':<12} {'log(Wk)':<10} {'Distribusi Cluster'}")
    print("-" * 55)
    
    for k in range(1, max_clusters + 1):
        # Clustering data asli
        link = linkage(dist_condensed, method='single')
        labels = fcluster(link, k, criterion='maxclust')
        
        # Hitung Wk
        Wk = compute_Wk(dist_matrix, labels)
        log_Wk = np.log(Wk) if Wk > 0 else 0.0
        
        Wks_original.append(Wk)
        log_Wks.append(log_Wk)
        
        # Hitung distribusi cluster
        unique, counts = np.unique(labels, return_counts=True)
        cluster_dist = dict(zip(unique, counts))
        
        print(f"{k:<3} {Wk:<12.6f} {log_Wk:<10.6f} {cluster_dist}")
    
    # 3. GENERATE DATA REFERENSI DAN PERHITUNGAN DISPERSI BOOTSTRAP
    print("\n" + "-"*50)
    print("3. DATA REFERENSI DAN DISPERSI BOOTSTRAP")
    print("-"*50)
    
    B = 50  # Jumlah bootstrap samples
    n_periods, n_assets = train.shape
    
    log_Wkbs_bootstrap = []
    
    print(f"\nMenghitung {B} bootstrap samples...")
    
    for b in range(B):
        # Generate reference data dari uniform distribution
        ref_returns = np.random.uniform(0, 1, size=(n_periods, n_assets))
        ref_returns_df = pd.DataFrame(ref_returns, columns=train.columns)
        
        # Hitung korelasi dan jarak untuk reference data
        ref_corr = ref_returns_df.corr()
        ref_dist = correl_dist(ref_corr)
        ref_condensed = squareform(ref_dist, checks=False)
        
        log_Wkbs = []
        
        for k in range(1, max_clusters + 1):
            # Clustering untuk reference data
            link_ref = linkage(ref_condensed, method='single')
            labels_ref = fcluster(link_ref, k, criterion='maxclust')
            
            Wkb = compute_Wk(ref_dist, labels_ref)
            log_Wkb = np.log(Wkb) if Wkb > 0 else 0.0
            log_Wkbs.append(log_Wkb)
        
        log_Wkbs_bootstrap.append(log_Wkbs)
    
    # 4. PERHITUNGAN GAP STATISTIC
    print("\n" + "-"*50)
    print("4. PERHITUNGAN GAP STATISTIC")
    print("-"*50)
    
    log_Wkbs_bootstrap = np.array(log_Wkbs_bootstrap)
    
    # E[log(W_kb)] = rata-rata bootstrap
    E_logWkb = np.mean(log_Wkbs_bootstrap, axis=0)
    
    # Gap(k) = E[log(W_kb)] - log(W_k)
    gaps = E_logWkb - np.array(log_Wks)
    
    # Hitung standard deviation
    sd_k = np.std(log_Wkbs_bootstrap, axis=0, ddof=1)
    s_k = sd_k * np.sqrt(1 + 1/B)
    
    print(f"\n{'k':<3} {'log(Wk)':<10} {'E[log(Wkb)]':<12} {'Gap(k)':<10} {'sd(k)':<10} {'s(k)':<10}")
    print("-" * 70)
    
    for k in range(len(gaps)):
        print(f"{k+1:<3} {log_Wks[k]:<10.6f} {E_logWkb[k]:<12.6f} {gaps[k]:<10.6f} {sd_k[k]:<10.6f} {s_k[k]:<10.6f}")
    
    # 5. PEMILIHAN JUMLAH CLUSTER OPTIMAL
    print("\n" + "-"*50)
    print("5. PEMILIHAN JUMLAH CLUSTER OPTIMAL")
    print("-"*50)
    
    optimal_k = 1
    found = False
    
    print("\nATURAN PEMILIHAN:")
    print("Pilih k terkecil dimana: Gap(k) ≥ Gap(k+1) - s(k+1)")
    print("-" * 60)
    
    for k in range(len(gaps) - 1):
        gap_k = gaps[k]
        gap_kplus1 = gaps[k+1]
        s_kplus1 = s_k[k+1]
        condition_value = gap_kplus1 - s_kplus1
        condition = gap_k >= condition_value
        
        print(f"k = {k+1}:")
        print(f"  Gap({k+1}) = {gap_k:.6f}")
        print(f"  Gap({k+2}) - s({k+2}) = {gap_kplus1:.6f} - {s_kplus1:.6f} = {condition_value:.6f}")
        print(f"  Kondisi: {gap_k:.6f} ≥ {condition_value:.6f} ? {condition}")
        
        if condition:
            optimal_k = k + 1
            found = True
            print(f"  ✓ KONDISI TERPENUHI! Optimal k = {optimal_k}")
            break
        else:
            print(f"  ✗ Kondisi tidak terpenuhi")
        print()
    
    if not found:
        optimal_k = np.argmax(gaps) + 1
        print(f"Tidak ada kondisi yang terpenuhi, pilih k dengan Gap terbesar: k = {optimal_k}")
    
    # 6. VISUALISASI HASIL
    print("\n" + "-"*50)
    print("6. VISUALISASI GAP STATISTIC")
    print("-"*50)
    
    plt.figure(figsize=(12, 10))
    
    k_values = range(1, max_clusters + 1)
    
    # Plot 1: Perbandingan log(Wk)
    plt.subplot(3, 1, 1)
    plt.plot(k_values, log_Wks, 'bo-', label='log(Wk) - Data Asli', linewidth=2, markersize=6)
    plt.plot(k_values, E_logWkb, 'ro-', label='E[log(Wkb)] - Reference', linewidth=2, markersize=6)
    plt.xlabel('Jumlah Cluster (k)')
    plt.ylabel('log(Wk)')
    plt.title('Perbandingan Dispersi: Data Asli vs Reference')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot 2: Gap Statistic
    plt.subplot(3, 1, 2)
    plt.plot(k_values, gaps, 'go-', label='Gap(k)', linewidth=2, markersize=8)
    plt.axvline(x=optimal_k, color='red', linestyle='--', 
                label=f'Optimal k = {optimal_k}', linewidth=2)
    
    # Error bars
    plt.errorbar(k_values, gaps, yerr=s_k, fmt='o', color='green', 
                 alpha=0.5, capsize=5, label='Gap(k) ± s(k)')
    
    plt.xlabel('Jumlah Cluster (k)')
    plt.ylabel('Gap(k)')
    plt.title('Gap Statistic dengan Error Bars')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot 3: Aturan pemilihan
    plt.subplot(3, 1, 3)
    
    # Hitung Gap(k+1) - s(k+1) untuk setiap k
    gap_minus_s = []
    for k in range(len(gaps)):
        if k < len(gaps) - 1:
            gap_minus_s.append(gaps[k+1] - s_k[k+1])
        else:
            gap_minus_s.append(np.nan)
    
    plt.plot(k_values[:-1], gaps[:-1], 'bo-', label='Gap(k)', linewidth=2)
    plt.plot(k_values[1:], gap_minus_s[:-1], 'ro-', label='Gap(k+1) - s(k+1)', linewidth=2)
    
    # Tandai titik optimal
    if found and optimal_k > 0 and optimal_k < len(gaps):
        plt.axvline(x=optimal_k, color='green', linestyle='--', 
                    label=f'Optimal: k={optimal_k}', linewidth=2)
    
    plt.xlabel('Jumlah Cluster (k)')
    plt.ylabel('Nilai')
    plt.title('Aturan Pemilihan: Gap(k) vs Gap(k+1)-s(k+1)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 7. VISUALISASI DENDROGRAM DENGAN PEMOTONGAN OPTIMAL
    print("\n" + "-"*50)
    print("7. VISUALISASI DENDROGRAM DENGAN PEMOTONGAN OPTIMAL")
    print("-"*50)
    
    # Buat linkage untuk visualisasi
    link_final = linkage(dist_condensed, method='single')
    
    # Hitung threshold untuk pemotongan berdasarkan jumlah cluster optimal
    last_merge_distances = link_final[:, 2]
    cut_threshold = last_merge_distances[-optimal_k]
    
    plt.figure(figsize=(14, 8))
    dendrogram(link_final, labels=train.columns.tolist(), leaf_rotation=90, leaf_font_size=8,color_threshold=0)
    
    # Tambahkan garis pemotongan
    plt.axhline(y=cut_threshold, color='red', linestyle='--', linewidth=3, 
                label=f'Cutoff Threshold (k={optimal_k})')
    
    plt.title(f'Dendrogram dengan Pemotongan Optimal (k={optimal_k}) - Gap Statistic')
    plt.xlabel('Assets')
    plt.ylabel('Distance')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"Threshold pemotongan: {cut_threshold:.4f}")
    print(f"Jumlah cluster yang dihasilkan: {optimal_k}")
    
    # 8. INTERPRETASI HASIL
    print("\n" + "-"*50)
    print("8. INTERPRETASI HASIL")
    print("-"*50)
    
    print(f"Jumlah cluster optimal: {optimal_k}")
    print(f"\nPenjelasan:")
    print(f"- Gap statistic mengukur seberapa baik clustering data asli dibandingkan")
    print(f"  dengan data referensi yang tidak memiliki struktur cluster")
    print(f"- Gap(k) yang besar menunjukkan clustering data asli lebih baik")
    print(f"- Aturan pemilihan: pilih k terkecil dimana Gap(k) ≥ Gap(k+1) - s(k+1)")
    print(f"- Ini berarti kita memilih k dimana penambahan cluster berikutnya")
    print(f"  tidak memberikan improvement yang signifikan")
    
    if optimal_k == 1:
        print(f"\n⚠️  Hasil: k=1 menunjukkan data mungkin tidak memiliki struktur cluster yang jelas")
    elif optimal_k == max_clusters:
        print(f"\n⚠️  Hasil: k={optimal_k} (maksimum) mungkin perlu meningkatkan max_clusters")
    else:
        print(f"\n✓ Hasil: k={optimal_k} menunjukkan struktur cluster yang optimal dalam data")
    
    return {
        'optimal_k': optimal_k,
        'gaps': gaps,
        's_k': s_k,
        'log_Wks': log_Wks,
        'E_logWkb': E_logWkb,
        'Wks_original': Wks_original
    }

gap_analysis = analisis_gap_statistic_mendalam()

# =======================================
# 12. VISUALISASI 15 TANGGAL TERAKHIR
# =======================================

def plot_last_15_days(results):
    """Plot 15 hari terakhir untuk return, drawdown, dan turnover"""
    
    if not results:
        print("No results available for plotting!")
        return
    
    print("\n" + "="*80)
    print("VISUALISASI 15 TANGGAL TERAKHIR")
    print("="*80)
    
    colors = plt.cm.tab20(np.linspace(0, 1, len(results)))
    
    # 1. PORTFOLIO VALUE - 15 HARI TERAKHIR
    plt.figure(figsize=(15, 10))
    
    plt.subplot(3, 1, 1)
    for i, (combo_name, metrics) in enumerate(results.items()):
        # Ambil 15 data terakhir
        last_15_days = metrics['final_portfolio'].tail(15)
        label = f"{metrics['linkage_method'].upper()}-L"
        
        plt.plot(last_15_days.index, last_15_days.values, 
                label=label, linewidth=2.5, color=colors[i], marker='o', markersize=4)
    
    plt.title("Portfolio Performance - 15 Hari Terakhir", fontsize=14, fontweight='bold')
    plt.ylabel("Portfolio Value")
    plt.xlabel("Date")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    
    # 2. DRAWDOWN - 15 HARI TERAKHIR
    plt.subplot(3, 1, 2)
    for i, (combo_name, metrics) in enumerate(results.items()):
        # Ambil 15 data terakhir dari drawdown
        last_15_drawdown = metrics['drawdown'].tail(15)
        label = f"{metrics['linkage_method'].upper()}-L"
        
        plt.plot(last_15_drawdown.index, last_15_drawdown.values, 
                label=label, linewidth=2, color=colors[i], marker='s', markersize=4)
    
    plt.title("Drawdown - 15 Hari Terakhir", fontsize=14, fontweight='bold')
    plt.ylabel("Drawdown")
    plt.xlabel("Date")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    
    # 3. TURNOVER - 15 PERIODE TERAKHIR
    plt.subplot(3, 1, 3)
    for i, (combo_name, metrics) in enumerate(results.items()):
        # Ambil 15 data terakhir dari turnover
        if len(metrics['turnovers']) >= 15:
            last_15_turnover = metrics['turnovers'][-15:]
        else:
            last_15_turnover = metrics['turnovers']
            
        # Buat index untuk turnover (periode rolling)
        turnover_index = range(len(last_15_turnover))
        label = f"{metrics['linkage_method'].upper()}-L"
        
        plt.plot(turnover_index, last_15_turnover, 
                label=label, linewidth=2, color=colors[i], marker='^', markersize=5)
    
    plt.title("Portfolio Turnover - 15 Periode Terakhir", fontsize=14, fontweight='bold')
    plt.ylabel("Turnover")
    plt.xlabel("Rolling Window (Terakhir)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

    # 4. DETAILED ANALYSIS - 15 HARI TERAKHIR
    print("\nDETAILED ANALYSIS - 15 HARI TERAKHIR:")
    print("-" * 80)
    
    for combo_name, metrics in results.items():
        print(f"\n{metrics['linkage_method'].upper()}-LINKAGE:")
        print("-" * 40)
        
        # Portfolio Value Analysis
        last_15_values = metrics['final_portfolio'].tail(15)
        if len(last_15_values) > 0:
            start_val = last_15_values.iloc[0]
            end_val = last_15_values.iloc[-1]
            total_return_15d = (end_val - start_val) / start_val
            
            print(f"Portfolio Value:")
            print(f"  Awal  : {start_val:.4f}")
            print(f"  Akhir : {end_val:.4f}")
            print(f"  Return: {total_return_15d:.2%}")
        
        # Drawdown Analysis
        last_15_drawdown = metrics['drawdown'].tail(15)
        if len(last_15_drawdown) > 0:
            max_drawdown_15d = last_15_drawdown.min()
            avg_drawdown_15d = last_15_drawdown.mean()
            
            print(f"Drawdown:")
            print(f"  Maksimum: {max_drawdown_15d:.2%}")
            print(f"  Rata-rata: {avg_drawdown_15d:.2%}")
        
        # Turnover Analysis
        if len(metrics['turnovers']) >= 15:
            last_15_turnover = metrics['turnovers'][-15:]
            avg_turnover_15d = np.mean(last_15_turnover)
            max_turnover_15d = np.max(last_15_turnover)
            
            print(f"Turnover:")
            print(f"  Rata-rata: {avg_turnover_15d:.4f}")
            print(f"  Maksimum: {max_turnover_15d:.4f}")
        
        print()

# 5. VISUALISASI KOMPARASI METRIK 15 HARI TERAKHIR
def plot_metrics_comparison_last_15(results):
    """Plot perbandingan metrik performance 15 hari terakhir"""
    
    if not results:
        return
    
    print("\n" + "="*80)
    print("PERBANDINGAN METRIK - 15 HARI TERAKHIR")
    print("="*80)
    
    # Prepare data for comparison
    methods = []
    returns_15d = []
    max_drawdowns_15d = []
    avg_turnovers_15d = []
    
    for combo_name, metrics in results.items():
        method_name = metrics['linkage_method'].upper()
        methods.append(method_name)
        
        # Calculate 15-day return
        last_15_values = metrics['final_portfolio'].tail(15)
        if len(last_15_values) > 1:
            start_val = last_15_values.iloc[0]
            end_val = last_15_values.iloc[-1]
            return_15d = (end_val - start_val) / start_val
        else:
            return_15d = 0
        returns_15d.append(return_15d)
        
        # Calculate 15-day max drawdown
        last_15_drawdown = metrics['drawdown'].tail(15)
        max_drawdown_15d = last_15_drawdown.min() if len(last_15_drawdown) > 0 else 0
        max_drawdowns_15d.append(max_drawdown_15d)
        
        # Calculate 15-day average turnover
        if len(metrics['turnovers']) >= 15:
            last_15_turnover = metrics['turnovers'][-15:]
            avg_turnover_15d = np.mean(last_15_turnover)
        else:
            avg_turnover_15d = np.mean(metrics['turnovers']) if metrics['turnovers'] else 0
        avg_turnovers_15d.append(avg_turnover_15d)
    
    # Create comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    colors = plt.cm.Set3(np.linspace(0, 1, len(methods)))
    
    # Plot 1: 15-day Returns
    axes[0, 0].bar(methods, returns_15d, color=colors, alpha=0.7)
    axes[0, 0].set_title('Return 15 Hari Terakhir', fontsize=14, fontweight='bold')
    axes[0, 0].set_ylabel('Return')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, v in enumerate(returns_15d):
        axes[0, 0].text(i, v, f'{v:.2%}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: 15-day Max Drawdown
    axes[0, 1].bar(methods, max_drawdowns_15d, color=colors, alpha=0.7)
    axes[0, 1].set_title('Maximum Drawdown 15 Hari Terakhir', fontsize=14, fontweight='bold')
    axes[0, 1].set_ylabel('Drawdown')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, v in enumerate(max_drawdowns_15d):
        axes[0, 1].text(i, v, f'{v:.2%}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 3: 15-day Average Turnover
    axes[1, 0].bar(methods, avg_turnovers_15d, color=colors, alpha=0.7)
    axes[1, 0].set_title('Rata-rata Turnover 15 Periode Terakhir', fontsize=14, fontweight='bold')
    axes[1, 0].set_ylabel('Turnover')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, v in enumerate(avg_turnovers_15d):
        axes[1, 0].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 4: Combined Performance Score
    # Simple scoring: higher return, lower drawdown, lower turnover = better
    returns_score = (np.array(returns_15d) - min(returns_15d)) / (max(returns_15d) - min(returns_15d) + 1e-10)
    drawdown_score = 1 - (np.array(max_drawdowns_15d) - min(max_drawdowns_15d)) / (max(max_drawdowns_15d) - min(max_drawdowns_15d) + 1e-10)
    turnover_score = 1 - (np.array(avg_turnovers_15d) - min(avg_turnovers_15d)) / (max(avg_turnovers_15d) - min(avg_turnovers_15d) + 1e-10)
    
    combined_score = (returns_score + drawdown_score + turnover_score) / 3
    
    axes[1, 1].bar(methods, combined_score, color=colors, alpha=0.7)
    axes[1, 1].set_title('Combined Performance Score (15 Hari)', fontsize=14, fontweight='bold')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, v in enumerate(combined_score):
        axes[1, 1].text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print performance ranking
    print("\nPERFORMANCE RANKING - 15 HARI TERAKHIR:")
    print("-" * 50)
    
    ranking_data = []
    for i, method in enumerate(methods):
        ranking_data.append({
            'Method': method,
            'Return_15d': returns_15d[i],
            'Max_Drawdown_15d': max_drawdowns_15d[i],
            'Avg_Turnover_15d': avg_turnovers_15d[i],
            'Combined_Score': combined_score[i]
        })
    
    ranking_df = pd.DataFrame(ranking_data)
    ranking_df = ranking_df.sort_values('Combined_Score', ascending=False)
    
    print(f"\n{'Rank':<4} {'Method':<12} {'Return':<10} {'Max DD':<10} {'Turnover':<10} {'Score':<8}")
    print("-" * 60)
    for i, (idx, row) in enumerate(ranking_df.iterrows()):
        print(f"{i+1:<4} {row['Method']:<12} {row['Return_15d']:>8.2%} {row['Max_Drawdown_15d']:>8.2%} {row['Avg_Turnover_15d']:>9.4f} {row['Combined_Score']:>7.3f}")

# 6. TABEL RINGKASAN PERFORMANCE 15 HARI TERAKHIR
def print_summary_table_last_15(results):
    """Print tabel ringkasan performance 15 hari terakhir"""
    
    if not results:
        return
    
    print("\n" + "="*80)
    print("TABEL RINGKASAN PERFORMANCE - 15 HARI TERAKHIR")
    print("="*80)
    
    summary_data = []
    
    for combo_name, metrics in results.items():
        # Portfolio Value
        last_15_values = metrics['final_portfolio'].tail(15)
        if len(last_15_values) > 1:
            start_val = last_15_values.iloc[0]
            end_val = last_15_values.iloc[-1]
            return_15d = (end_val - start_val) / start_val
            volatility_15d = last_15_values.pct_change().std()
        else:
            return_15d = 0
            volatility_15d = 0
        
        # Drawdown
        last_15_drawdown = metrics['drawdown'].tail(15)
        max_drawdown_15d = last_15_drawdown.min() if len(last_15_drawdown) > 0 else 0
        avg_drawdown_15d = last_15_drawdown.mean() if len(last_15_drawdown) > 0 else 0
        
        # Turnover
        if len(metrics['turnovers']) >= 15:
            last_15_turnover = metrics['turnovers'][-15:]
            avg_turnover_15d = np.mean(last_15_turnover)
            max_turnover_15d = np.max(last_15_turnover)
        else:
            avg_turnover_15d = np.mean(metrics['turnovers']) if metrics['turnovers'] else 0
            max_turnover_15d = np.max(metrics['turnovers']) if metrics['turnovers'] else 0
        
        summary_data.append({
            'Linkage_Method': metrics['linkage_method'].upper(),
            'Return_15d': f"{return_15d:.2%}",
            'Volatility_15d': f"{volatility_15d:.2%}" if volatility_15d > 0 else "N/A",
            'Max_Drawdown_15d': f"{max_drawdown_15d:.2%}",
            'Avg_Drawdown_15d': f"{avg_drawdown_15d:.2%}",
            'Avg_Turnover_15d': f"{avg_turnover_15d:.4f}",
            'Max_Turnover_15d': f"{max_turnover_15d:.4f}"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))

# HERC

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from scipy.stats import skew, kurtosis
import warnings
import seaborn as sns
warnings.filterwarnings('ignore')

# =======================================
# 1. FUNGSI UTILITAS DASAR
# =======================================

def correl_dist(corr):
    return np.sqrt(0.5 * (1 - corr))

def compute_Wk(dist_matrix, labels):
    if isinstance(dist_matrix, pd.DataFrame):
        dist_matrix = dist_matrix.values
    
    clusters = np.unique(labels)
    Wk = 0.0
    for c in clusters:
        idx = np.where(labels == c)[0]
        if len(idx) > 1:
            sub_dist = dist_matrix[idx][:, idx]
            Wk += np.sum(sub_dist) / (2 * len(idx))
    return Wk

def gap_statistic_corrected(returns_data, max_clusters=20, B=100, method='ward', random_seed=42):
    np.random.seed(random_seed)
    
    n_periods, n_assets = returns_data.shape
    max_clusters = min(max_clusters, n_assets - 1)
    
    if max_clusters < 2:
        return 2
    
    # 1. Hitung untuk data asli
    corr_matrix = returns_data.corr()
    dist_matrix = correl_dist(corr_matrix)
    dist_condensed = squareform(dist_matrix, checks=False)
    
    log_Wks = []
    for k in range(1, max_clusters + 1):
        link = linkage(dist_condensed, method=method)
        labels = fcluster(link, k, criterion='maxclust')
        Wk = compute_Wk(dist_matrix, labels)
        log_Wks.append(np.log(Wk) if Wk > 0 else 0.0)
    
    # 2. Hitung untuk data referensi
    log_Wkbs_bootstrap = []
    
    for b in range(B):
        ref_returns = np.random.uniform(0, 1, size=(n_periods, n_assets))
        ref_returns_df = pd.DataFrame(ref_returns, columns=returns_data.columns)
        
        ref_corr = ref_returns_df.corr()
        ref_dist = correl_dist(ref_corr)
        ref_condensed = squareform(ref_dist, checks=False)
        
        log_Wkbs = []
        for k in range(1, max_clusters + 1):
            link_ref = linkage(ref_condensed, method=method)
            labels_ref = fcluster(link_ref, k, criterion='maxclust')
            Wkb = compute_Wk(ref_dist, labels_ref)
            log_Wkb = np.log(Wkb) if Wkb > 0 else 0.0
            log_Wkbs.append(log_Wkb)
        
        log_Wkbs_bootstrap.append(log_Wkbs)
    
    # 3. Hitung Gap Statistic
    log_Wkbs_bootstrap = np.array(log_Wkbs_bootstrap)
    E_logWkb = np.mean(log_Wkbs_bootstrap, axis=0)
    gaps = E_logWkb - np.array(log_Wks)
    
    # 4. Hitung standard deviation
    sd_k = np.std(log_Wkbs_bootstrap, axis=0, ddof=1)
    s_k = sd_k * np.sqrt(1 + 1/B)
    
    # 5. Pilih jumlah cluster optimal
    optimal_k = 1
    for k in range(len(gaps)-1):
        if k+1 < len(s_k) and gaps[k] >= gaps[k+1] - s_k[k+1]:
            optimal_k = k + 1
            break
    
    if optimal_k == 1 and len(gaps) > 0:
        optimal_k = np.argmax(gaps) + 1
    
    optimal_k = max(1, min(optimal_k, max_clusters))
    
    return optimal_k

# =======================================
# 2. FUNGSI RISK MEASURE
# =======================================

def compute_risk_measure(returns, risk_measure="variance", alpha=0.05):
    returns = np.asarray(returns).ravel()
    
    if risk_measure == "variance":
        return np.var(returns, ddof=1)
    
    elif risk_measure == "cvar":
        if returns.size == 0:
            return 0.0
        q = np.quantile(returns, alpha)
        tail = returns[returns <= q]
        return -tail.mean() if tail.size > 0 else -q
    
    elif risk_measure == "cdar":
        s = pd.Series(returns)
        cum = (1 + s).cumprod()
        running_max = cum.cummax()
        drawdowns = (running_max - cum) / running_max
        
        if drawdowns.empty:
            return 0.0
        
        q = np.quantile(drawdowns, 1 - alpha)
        tail = drawdowns[drawdowns >= q]
        return tail.mean() if tail.size > 0 else q
    
    else:
        return np.var(returns, ddof=1)

def compute_covariance_matrix(returns, risk_measure="variance", alpha=0.05):
    n_assets = returns.shape[1]
    
    if risk_measure == "variance":
        return returns.cov().values
    else:
        return returns.cov().values

# =======================================
# 3. IMPLEMENTASI HERC CORE
# =======================================

def compute_cluster_risk_contribution(cluster_assets, returns, cov_matrix, risk_measure="variance", alpha=0.05):
    if len(cluster_assets) == 0:
        return 0.0
    
    # 1. Hitung bobot dengan Naive Risk Parity
    individual_risks = []
    for asset_idx in cluster_assets:
        asset_returns = returns.iloc[:, asset_idx]
        
        if risk_measure == "variance":
            risk = np.var(asset_returns, ddof=1)
        elif risk_measure == "cvar":
            q = np.quantile(asset_returns, alpha)
            tail_returns = asset_returns[asset_returns <= q]
            risk = -tail_returns.mean() if len(tail_returns) > 0 else -q
        elif risk_measure == "cdar":
            s = pd.Series(asset_returns)
            cum = (1 + s).cumprod()
            running_max = cum.cummax()
            drawdowns = (running_max - cum) / running_max
            q = np.quantile(drawdowns, 1 - alpha) if not drawdowns.empty else 0
            tail_drawdowns = drawdowns[drawdowns >= q]
            risk = tail_drawdowns.mean() if len(tail_drawdowns) > 0 else q
        else:
            risk = np.var(asset_returns, ddof=1)
        
        individual_risks.append(max(risk, 1e-12))
    
    # 2. Naive Risk Parity weights
    inv_risks = [1.0 / r for r in individual_risks]
    total_inv_risk = sum(inv_risks)
    
    if total_inv_risk <= 0:
        weights = [1.0 / len(cluster_assets)] * len(cluster_assets)
    else:
        weights = [inv_r / total_inv_risk for inv_r in inv_risks]
    
    weights_array = np.array(weights)
    
    # 3. Hitung risk contribution
    if risk_measure == "variance":
        cluster_cov = cov_matrix[np.ix_(cluster_assets, cluster_assets)]
        port_variance = weights_array.T @ cluster_cov @ weights_array
        
        if port_variance <= 0:
            return np.sum(weights_array)
        
        marginal_contrib = cluster_cov @ weights_array
        risk_contributions = (weights_array * marginal_contrib) / port_variance
    
    elif risk_measure == "cvar":
        # Simplified CVaR contribution
        cluster_returns = returns.iloc[:, cluster_assets]
        port_returns = cluster_returns @ weights_array
        
        # Gunakan boolean mask sebagai integer index
        extreme_loss_mask = (port_returns <= np.quantile(port_returns, alpha))
        extreme_indices = np.where(extreme_loss_mask)[0]
        
        if len(extreme_indices) == 0:
            conditional_expectations = [-np.mean(returns.iloc[:, asset_idx]) for asset_idx in cluster_assets]
        else:
            conditional_expectations = []
            for asset_idx in cluster_assets:
                # Ambil returns berdasarkan index yang valid
                cond_exp = np.mean(returns.iloc[extreme_indices, asset_idx])
                conditional_expectations.append(-cond_exp)
        
        risk_contributions = weights_array * np.array(conditional_expectations)
    
    elif risk_measure == "cdar":
        cluster_returns = returns.iloc[:, cluster_assets].values
        w = weights_array.copy()
        h = 1e-5  # step kecil finite difference

        # -----------------------------
        # fungsi CDaR portofolio
        # -----------------------------
        def compute_cdar_portfolio(weights):
            port_returns = cluster_returns @ weights
            cum = np.cumprod(1 + port_returns)
            running_max = np.maximum.accumulate(cum)
            drawdowns = (running_max - cum) / running_max

            if len(drawdowns) == 0:
                return 0.0

            q = np.quantile(drawdowns, 1 - alpha)
            tail = drawdowns[drawdowns >= q]
            return np.mean(tail) if len(tail) > 0 else q

        # CDaR baseline
        cdar_0 = compute_cdar_portfolio(w)

        cdar_contributions = []

        # -----------------------------
        # Euler risk contribution
        # via central finite difference
        # -----------------------------
        for i in range(len(w)):
            dw = np.zeros_like(w)
            dw[i] = h

            cdar_plus = compute_cdar_portfolio(w + dw)
            cdar_minus = compute_cdar_portfolio(w - dw)

            # turunan parsial numerik
            marginal_cdar = (cdar_plus - cdar_minus) / (2 * h)

            # Euler contribution
            rc_i = w[i] * marginal_cdar
            cdar_contributions.append(rc_i)

        risk_contributions = np.array(cdar_contributions)
        
    else:
        cluster_cov = cov_matrix[np.ix_(cluster_assets, cluster_assets)]
        port_variance = weights_array.T @ cluster_cov @ weights_array
        if port_variance <= 0:
            return np.sum(weights_array)
        marginal_contrib = cluster_cov @ weights_array
        risk_contributions = (weights_array * marginal_contrib) / port_variance
    
    total_rc = np.sum(risk_contributions)
    return max(total_rc, 1e-12)

def get_all_assets_from_node(node_idx, linkage_matrix, n_total_assets):
    """Rekursif mendapatkan semua assets dari node dalam dendrogram"""
    if node_idx < n_total_assets:
        return [node_idx]
    
    idx = node_idx - n_total_assets
    left = int(linkage_matrix[idx, 0])
    right = int(linkage_matrix[idx, 1])
    
    return (get_all_assets_from_node(left, linkage_matrix, n_total_assets) + 
            get_all_assets_from_node(right, linkage_matrix, n_total_assets))

def find_cutting_point(linkage_matrix, assets, n_total_assets):
    """Cari titik potong dalam dendrogram untuk membagi assets menjadi dua cluster"""
    assets_set = set(assets)
    
    for i in range(len(linkage_matrix)):
        node_idx = i + n_total_assets
        
        left_idx = int(linkage_matrix[i, 0])
        right_idx = int(linkage_matrix[i, 1])
        
        left_assets = get_all_assets_from_node(left_idx, linkage_matrix, n_total_assets)
        right_assets = get_all_assets_from_node(right_idx, linkage_matrix, n_total_assets)
        
        all_node_assets = sorted(left_assets + right_assets)
        
        if sorted(assets) == all_node_assets:
            left_assets_filtered = [a for a in left_assets if a in assets_set]
            right_assets_filtered = [a for a in right_assets if a in assets_set]
            
            if left_assets_filtered and right_assets_filtered:
                return left_assets_filtered, right_assets_filtered
    
    # Fallback
    mid = len(assets) // 2
    return assets[:mid], assets[mid:]

def herc_recursive_division(linkage_matrix, assets, returns, cov_matrix, 
                           current_level=0, risk_measure="variance", alpha=0.05):
    """HERC dengan recursive division"""
    
    n_assets = len(assets)
    n_total_assets = len(returns.columns)
    
    # Base case 1: 1 asset
    if n_assets == 1:
        return {assets[0]: 1.0}
    
    # Base case 2: 2 assets
    if n_assets == 2:
        asset1, asset2 = assets
        
        risk1 = compute_risk_measure(returns.iloc[:, asset1], risk_measure, alpha)
        risk2 = compute_risk_measure(returns.iloc[:, asset2], risk_measure, alpha)
        
        risk1 = max(risk1, 1e-12)
        risk2 = max(risk2, 1e-12)
        
        inv_risk1 = 1.0 / risk1
        inv_risk2 = 1.0 / risk2
        total_inv_risk = inv_risk1 + inv_risk2
        
        weights = {
            asset1: inv_risk1 / total_inv_risk,
            asset2: inv_risk2 / total_inv_risk
        }
        return weights
    
    # Top-down recursive division
    left_cluster, right_cluster = find_cutting_point(linkage_matrix, assets, n_total_assets)
    
    if not left_cluster or not right_cluster:
        mid = n_assets // 2
        left_cluster = assets[:mid]
        right_cluster = assets[mid:]
    
    # Hitung risk contribution
    rc_left = compute_cluster_risk_contribution(left_cluster, returns, cov_matrix, risk_measure, alpha)
    rc_right = compute_cluster_risk_contribution(right_cluster, returns, cov_matrix, risk_measure, alpha)
    
    # Tentukan bobot antar cluster
    if rc_left + rc_right > 0:
        weight_left = rc_left / (rc_left + rc_right)
        weight_right = rc_right / (rc_left + rc_right)
    else:
        weight_left = len(left_cluster) / n_assets
        weight_right = len(right_cluster) / n_assets
    
    # Rekursi
    weights_left = herc_recursive_division(linkage_matrix, left_cluster, returns, cov_matrix, 
                                          current_level + 1, risk_measure, alpha)
    weights_right = herc_recursive_division(linkage_matrix, right_cluster, returns, cov_matrix, 
                                           current_level + 1, risk_measure, alpha)
    
    # Gabungkan bobot
    final_weights = {}
    
    for asset, weight in weights_left.items():
        final_weights[asset] = weight * weight_left
    
    for asset, weight in weights_right.items():
        final_weights[asset] = weight * weight_right
    
    # Normalisasi
    weight_sum = sum(final_weights.values())
    if weight_sum > 0:
        final_weights = {asset: w/weight_sum for asset, w in final_weights.items()}
    
    return final_weights

def herc_top_down_recursive_division(returns, n_clusters, linkage_method='ward', 
                                    risk_measure="variance", alpha=0.05):
    """HERC dengan Top-down Recursive Division"""
    
    # 1. Hitung korelasi dan distance
    corr = returns.corr()
    dist = correl_dist(corr)
    dist = np.nan_to_num(dist, nan=0.0, posinf=0.0, neginf=0.0)
    
    # 2. Hierarchical clustering
    dist_condensed = squareform(dist, checks=False)
    link = linkage(dist_condensed, method=linkage_method)
    
    n_assets = len(returns.columns)
    assets = list(range(n_assets))
    
    # 3. Cluster labels untuk visualisasi
    cluster_labels = fcluster(link, n_clusters, criterion='maxclust')
    
    clusters = {}
    for i, asset_idx in enumerate(assets):
        cluster_id = cluster_labels[i]
        if cluster_id not in clusters:
            clusters[cluster_id] = []
        clusters[cluster_id].append(returns.columns[asset_idx])
    
    cluster_list = list(clusters.values())
    
    # 4. Hitung covariance matrix
    cov_matrix = compute_covariance_matrix(returns, risk_measure, alpha)
    
    # 5. Jalankan HERC recursive division
    final_weights_idx = herc_recursive_division(link, assets, returns, cov_matrix, 
                                               0, risk_measure, alpha)
    
    # 6. Konversi ke nama asset
    final_weights = {}
    for asset_idx, weight in final_weights_idx.items():
        asset_name = returns.columns[asset_idx]
        final_weights[asset_name] = weight
    
    # 7. Normalisasi
    weight_sum = sum(final_weights.values())
    if weight_sum > 0:
        final_weights = {asset: weight/weight_sum for asset, weight in final_weights.items()}
    
    weights_series = pd.Series(final_weights)
    
    return weights_series, link, cluster_list

# =======================================
# 4. ROLLING WINDOW IMPLEMENTATION
# =======================================

def precompute_optimal_clusters_all_linkages(returns, window_train=254, step=1, max_clusters=20):
    """Pre-compute optimal clusters untuk semua linkage methods"""
    print("Pre-computing optimal clusters for all linkage methods...")
    
    linkage_methods = ['ward', 'complete', 'average', 'single']
    optimal_clusters_dict = {}
    
    n_rolls = len(returns) - window_train
    
    for linkage_method in linkage_methods:
        cluster_counts = []
        print(f"  Computing for {linkage_method} linkage...")
        
        for t in range(0, n_rolls, step):
            if t + window_train >= len(returns):
                break
                
            train_data = returns.iloc[t:t + window_train]
            
            try:
                n_clusters = gap_statistic_corrected(
                    train_data,
                    max_clusters=min(max_clusters, len(train_data.columns)-1),
                    method=linkage_method,
                    random_seed=42,
                    B=50
                )
                n_clusters = max(2, n_clusters)
                cluster_counts.append(n_clusters)
                
            except Exception as e:
                cluster_counts.append(2)
                continue
        
        optimal_clusters_dict[linkage_method] = cluster_counts
        print(f"    {linkage_method}: {len(cluster_counts)} windows computed")
    
    return optimal_clusters_dict

def run_rolling_herc_consistent(returns, linkage_method='ward', risk_measure='variance', 
                               alpha=0.05, step=1, precomputed_clusters=None):
    """Jalankan HERC dengan cluster yang konsisten"""
    
    window_train = 254
    n_rolls = len(returns) - window_train
    
    portfolio_values = [1.0]
    weights_all = []
    portfolio_returns = []
    cluster_counts = []

    print(f"Running HERC: {linkage_method.upper()} linkage + {risk_measure.upper()} risk measure")
    
    for t in range(0, n_rolls, step):
        if t + window_train + 1 >= len(returns):
            break
            
        train_data = returns.iloc[t:t + window_train]
        test_data = returns.iloc[t + window_train:t + window_train + 1]
        
        if test_data.empty or len(train_data) < window_train:
            continue

        try:
            # 1. Gunakan pre-computed clusters
            if precomputed_clusters and linkage_method in precomputed_clusters:
                window_idx = t // step
                if window_idx < len(precomputed_clusters[linkage_method]):
                    n_clusters = precomputed_clusters[linkage_method][window_idx]
                else:
                    n_clusters = 2
            else:
                n_clusters = gap_statistic_corrected(
                    train_data, 
                    max_clusters=min(8, len(train_data.columns)-1),
                    method=linkage_method,
                    random_seed=42,
                    B=50
                )
                n_clusters = max(2, n_clusters)
            
            cluster_counts.append(n_clusters)

            # 2. Dapatkan weights dengan HERC
            weights, linkage_matrix, clusters = herc_top_down_recursive_division(
                train_data, 
                n_clusters=n_clusters,
                linkage_method=linkage_method,
                risk_measure=risk_measure,
                alpha=alpha
            )

            # Normalisasi
            weights = weights / weights.sum()
            weights_all.append(weights)

            # 3. Hitung portfolio return
            port_return = (test_data.values.flatten() @ weights.values).item()
            portfolio_returns.append(port_return)
            
            # 4. Update portfolio value
            new_value = portfolio_values[-1] * (1 + port_return)
            portfolio_values.append(new_value)
            
        except Exception as e:
            print(f"Error in window {t}: {e}")
            # Fallback: equal weighting
            n_assets = len(train_data.columns)
            weights = pd.Series(1.0/n_assets, index=train_data.columns)
            weights_all.append(weights)
            cluster_counts.append(2)
            port_return = test_data.mean().mean()
            portfolio_returns.append(port_return)
            new_value = portfolio_values[-1] * (1 + port_return)
            portfolio_values.append(new_value)
            continue

    return {
        'portfolio_values': portfolio_values,
        'weights_all': weights_all,
        'portfolio_returns': portfolio_returns,
        'cluster_counts': cluster_counts
    }

# =======================================
# 5. PERFORMANCE METRICS
# =======================================

def calculate_performance_metrics(results, returns, linkage_method, risk_measure):
    """Hitung semua metrik performance"""
    
    portfolio_values = results['portfolio_values']
    weights_all = results['weights_all']
    portfolio_returns = results['portfolio_returns']
    
    if len(portfolio_values) <= 1:
        return None
    
    # Time series portfolio values
    final_portfolio = pd.Series(portfolio_values[1:], 
                               index=returns.index[254:254 + len(portfolio_values[1:])])
    
    # Return series
    port_returns_series = pd.Series(portfolio_returns, 
                                   index=returns.index[254:254 + len(portfolio_returns)])
    
    # Turnover
    turnovers = []
    for i in range(1, len(weights_all)):
        turnover = np.sum(np.abs(weights_all[i] - weights_all[i-1]))
        turnovers.append(turnover)
    avg_turnover = np.mean(turnovers) if turnovers else 0

    # Sum of Squared Portfolio Weights
    sspw = np.mean([np.sum(w ** 2) for w in weights_all]) if weights_all else 0

    # Sharpe Ratio
    rf = 0.000110177  # risk-free rate
    if len(port_returns_series) > 1:
        sharpe_ratio = (port_returns_series.mean() - rf) / port_returns_series.std()
    else:
        sharpe_ratio = 0

    # Adjusted Sharpe Ratio
    if len(port_returns_series) > 1:
        S = skew(port_returns_series)
        K = kurtosis(port_returns_series, fisher=False)
        adjusted_sharpe = sharpe_ratio * (1 + (S / 6) * sharpe_ratio - ((K - 3) / 24) * sharpe_ratio**2)
    else:
        adjusted_sharpe = 0

    # Certainty Equivalent Return
    gamma = 1
    if len(port_returns_series) > 1:
        mean_return = port_returns_series.mean()
        variance_return = port_returns_series.var()
        cer = (mean_return - rf) - (0.5 * gamma * variance_return)
    else:
        cer = 0

    # Maximum Drawdown
    running_max = final_portfolio.cummax()
    drawdown = (final_portfolio - running_max) / running_max
    max_drawdown = drawdown.min()

    # Total Return
    total_return = final_portfolio.iloc[-1] - 1 if len(final_portfolio) > 0 else 0

    return {
        'final_portfolio': final_portfolio,
        'port_returns_series': port_returns_series,
        'drawdown': drawdown,
        'turnovers': turnovers,
        'total_return': total_return,
        'max_drawdown': max_drawdown,
        'avg_turnover': avg_turnover,
        'sspw': sspw,
        'sharpe_ratio': sharpe_ratio,
        'adjusted_sharpe': adjusted_sharpe,
        'cer': cer,
        'linkage_method': linkage_method,
        'risk_measure': risk_measure,
        'cluster_counts': results['cluster_counts']
    }

# =======================================
# 6. MAIN EXECUTION
# =======================================

def main_herc_analysis(prices):
    """Analisis HERC utama"""
    # 1. Baca data
    returns = prices.pct_change().dropna()
    
    print("Data return shape:", returns.shape)
    print(returns.head())
    print("\n" + "="*80)
    print("HERC COMPREHENSIVE ANALYSIS")
    print("="*80)
    
    # 2. Definisikan parameter
    linkage_methods = ['ward', 'complete', 'average', 'single']
    risk_measures = ['variance', 'cvar', 'cdar']
    alpha = 0.05
    
    # 3. Pre-compute clusters
    precomputed_clusters = precompute_optimal_clusters_all_linkages(
        returns, window_train=254, step=1, max_clusters=20
    )
    
    # 4. Jalankan HERC untuk semua kombinasi
    all_results = {}
    
    for linkage_method in linkage_methods:
        for risk_measure in risk_measures:
            combo_name = f"{linkage_method}_{risk_measure}"
            
            print(f"\nProcessing: {combo_name.upper()}")
            
            try:
                rolling_results = run_rolling_herc_consistent(
                    returns,
                    linkage_method=linkage_method,
                    risk_measure=risk_measure,
                    alpha=alpha,
                    precomputed_clusters=precomputed_clusters
                )
                
                metrics = calculate_performance_metrics(
                    rolling_results, returns, linkage_method, risk_measure
                )
                
                if metrics and len(rolling_results['portfolio_values']) > 5:
                    all_results[combo_name] = metrics
                    print(f"✓ SUCCESS: {combo_name}")
                    print(f"  Final Value: {metrics['final_portfolio'].iloc[-1]:.4f}")
                    print(f"  Total Return: {metrics['total_return']:.2%}")
                    print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.4f}")
                else:
                    print(f"✗ SKIPPED: {combo_name} - insufficient data")
                    
            except Exception as e:
                print(f"✗ ERROR: {combo_name} - {str(e)}")
                continue
    
    print(f"\nCompleted {len(all_results)} out of {len(linkage_methods)*len(risk_measures)} combinations")
    
    return all_results, returns

# =======================================
# 7. VISUALISASI
# =======================================

def plot_herc_results(all_results):
    """Plot hasil HERC"""
    if not all_results:
        print("No results to plot!")
        return
    
    # 1. Performance semua kombinasi
    plt.figure(figsize=(16, 10))
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_results)))
    
    for i, (combo_name, metrics) in enumerate(all_results.items()):
        label = f"{metrics['linkage_method'].upper()}-L/{metrics['risk_measure'].upper()}-R"
        plt.plot(metrics['final_portfolio'].index, metrics['final_portfolio'].values, 
                label=label, linewidth=2, color=colors[i])
    
    plt.title("HERC Portfolio Performance - All Combinations", fontsize=14)
    plt.ylabel("Portfolio Value")
    plt.xlabel("Date")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # 2. Drawdown semua kombinasi
    plt.figure(figsize=(16, 8))
    
    for i, (combo_name, metrics) in enumerate(all_results.items()):
        label = f"{metrics['linkage_method'].upper()}-L/{metrics['risk_measure'].upper()}-R"
        plt.plot(metrics['drawdown'].index, metrics['drawdown'].values, 
                label=label, linewidth=1.5, color=colors[i])
    
    plt.title("Drawdown Comparison - All Method Combinations", fontsize=14)
    plt.ylabel("Drawdown")
    plt.xlabel("Date")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # =======================================
    # 8. GRAFIK TURNOVER SEMUA KOMBINASI
    # =======================================
    plt.figure(figsize=(16, 8))
    
    for i, (combo_name, metrics) in enumerate(all_results.items()):
        label = f"{metrics['linkage_method'].upper()}-L/{metrics['risk_measure'].upper()}-R"
        plt.plot(metrics['turnovers'], label=label, linewidth=1.5, color=colors[i], marker='o', markersize=3)
    
    plt.title("Portfolio Turnover Over Time - All Method Combinations", fontsize=14)
    plt.ylabel("Turnover")
    plt.xlabel("Rolling Window")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # =======================================
    # 9. GRAFIK CLUSTER EVOLUTION SEMUA KOMBINASI
    # =======================================
    plt.figure(figsize=(16, 8))
    
    # Kelompokkan berdasarkan linkage method
    linkage_data = {}
    for combo_name, metrics in all_results.items():
        linkage_method = metrics['linkage_method']
        if linkage_method not in linkage_data:
            linkage_data[linkage_method] = metrics['cluster_counts']
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(linkage_data)))
    
    for i, (linkage_method, clusters) in enumerate(linkage_data.items()):
        plt.plot(range(len(clusters)), clusters, label=f"{linkage_method.upper()}-Linkage", 
                linewidth=2, marker='s', markersize=4, alpha=0.8, color=colors[i])
    
    plt.title("Cluster Evolution Over Time - By Linkage Method (NOW SAME AS HCAA)", fontsize=14)
    plt.ylabel("Number of Clusters")
    plt.xlabel("Rolling Window")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # 3. Tabel perbandingan
    print("\n" + "="*100)
    print("COMPREHENSIVE COMPARISON: HERC WITH DIFFERENT LINKAGE & RISK MEASURE COMBINATIONS")
    print("="*100)
    
    comparison_data = []
    for combo_name, metrics in all_results.items():
        comparison_data.append({
            'Linkage': metrics['linkage_method'].upper(),
            'Risk_Measure': metrics['risk_measure'].upper(),
            'Total_Return': f"{metrics['total_return']:.2%}",
            'Max_Drawdown': f"{metrics['max_drawdown']:.2%}",
            'Sharpe_Ratio': f"{metrics['sharpe_ratio']:.4f}",
            'Adj_Sharpe': f"{metrics['adjusted_sharpe']:.4f}",
            'CER': f"{metrics['cer']:.6f}",
            'Avg Turnover': f"{metrics['avg_turnover']:.4f}",
            'SSPW': f"{metrics['sspw']:.4f}",
            'Avg_Clusters': f"{np.mean(metrics['cluster_counts']):.1f}"
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df['Sharpe_Value'] = comparison_df['Sharpe_Ratio'].str.replace('%', '').astype(float)
    comparison_df = comparison_df.sort_values('Sharpe_Value', ascending=False)
    comparison_df = comparison_df.drop('Sharpe_Value', axis=1)
    
    print(comparison_df.to_string(index=False))
    
    # 4. Identifikasi kombinasi terbaik
    if all_results:
        best_combo_sharpe = max(all_results.keys(), key=lambda x: all_results[x]['sharpe_ratio'])
        best_metrics = all_results[best_combo_sharpe]
        
        print(f"\n{'='*80}")
        print(f"BEST PERFORMING COMBINATION BY SHARPE RATIO: {best_combo_sharpe.upper()}")
        print(f"{'='*80}")
        
        print(f"Linkage Method: {best_metrics['linkage_method'].upper()}")
        print(f"Risk Measure: {best_metrics['risk_measure'].upper()}")
        print(f"Total Return: {best_metrics['total_return']:.2%}")
        print(f"Maximum Drawdown: {best_metrics['max_drawdown']:.2%}")
        print(f"Sharpe Ratio: {best_metrics['sharpe_ratio']:.4f}")
        print(f"Adjusted Sharpe Ratio: {best_metrics['adjusted_sharpe']:.4f}")
        print(f"Certainty Equivalent Return (CER): {best_metrics['cer']:.6f}")
        print(f"Average Turnover: {best_metrics['avg_turnover']:.4f}")
        print(f"SSPW: {best_metrics['sspw']:.4f}")
        print(f"Average Clusters: {np.mean(best_metrics['cluster_counts']):.1f}")

# =======================================
# 8. ANALISIS WINDOW PERTAMA 
# =======================================

def compute_risk_contribution(weights, cov_matrix):
    weights = np.asarray(weights)
    port_variance = weights.T @ cov_matrix @ weights
    
    if port_variance <= 0:
        return np.zeros_like(weights)
    
    marginal_contrib = cov_matrix @ weights
    risk_contributions = (weights * marginal_contrib) / port_variance
    
    return risk_contributions

def analyze_first_window_herc_cvar(returns):
    print("\n" + "="*80)
    print("ANALISIS WINDOW PERTAMA - HERC CVaR AVERAGE LINKAGE")
    print("="*80)
    
    # 1. DATA PREPARATION
    window_train = 254
    train = returns.iloc[0:window_train]
    test = returns.iloc[window_train:window_train + 1]
    
    print(f"1. Data Preparation:")
    print(f"   Training period: {len(train)} days")
    print(f"   Number of assets: {len(train.columns)}")
    
    # 2. KORELASI DAN DISTANCE MATRIX
    print("\n2. Korelasi dan Distance Matrix")
    corr_matrix = train.corr()
    dist_matrix = correl_dist(corr_matrix)
    corr_values = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]
    print(f"   Average correlation: {corr_values.mean():.4f}")
    print(f"   Min correlation: {corr_values.min():.4f}")
    print(f"   Max correlation: {corr_values.max():.4f}")
    
    # Visualisasi korelasi
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0, 
                xticklabels=True, yticklabels=True, cbar_kws={'shrink': 0.8})
    plt.title('Correlation Matrix - First Window')
    plt.tight_layout()
    plt.show()
    
    # 3. HIERARCHICAL CLUSTERING
    print("\n3. Hierarchical Clustering")
    dist_condensed = squareform(dist_matrix, checks=False)
    link = linkage(dist_condensed, method='average')  # Simpan ke variabel 'link'
    
    # Dendrogram awal
    plt.figure(figsize=(14, 8))
    dendrogram(link, labels=train.columns.tolist(), leaf_rotation=90, leaf_font_size=8, color_threshold=0)
    plt.title('Dendrogram - Average Linkage (Full)')
    plt.xlabel('Assets')
    plt.ylabel('Distance')
    plt.tight_layout()
    plt.show()
    
    # 4. GAP STATISTIC
    print("\n4. Gap Statistic Calculation")
    optimal_k = gap_statistic_corrected(
        train, 
        max_clusters=min(15, len(train.columns)-1),
        method='average',
        random_seed=42,
        B=30
    )
    print(f"   Optimal number of clusters: {optimal_k}")
    
    # 5. HERC WEIGHTS CALCULATION
    print("\n5. HERC Weights Calculation (CVaR Risk Measure)")
    herc_weights, link_matrix, clusters = herc_top_down_recursive_division(
        train, 
        n_clusters=optimal_k, 
        linkage_method='average', 
        risk_measure='cvar', 
        alpha=0.05
    )
    
    # 6. WEIGHTS DISTRIBUTION
    print("\n6. Weights Distribution")
    print(f"   {'Asset':<15} {'Weight':<10} {'Percentage':<10}")
    print("   " + "-" * 35)
    
    # Sort weights descending
    sorted_weights = herc_weights.sort_values(ascending=False)
    for asset, weight in sorted_weights.items():
        print(f"   {asset:<15} {weight:<10.4f} {weight*100:<9.2f}%")
    
    # Visualisasi bobot
    plt.figure(figsize=(12, 6))
    sorted_weights.plot(kind='bar', color='steelblue', alpha=0.7)
    plt.title('HERC Portfolio Weights - First Window')
    plt.xlabel('Assets')
    plt.ylabel('Weight')
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    # 7. CLUSTER ANALYSIS
    print("\n7. Cluster Analysis")
    cluster_labels = fcluster(link, optimal_k, criterion='maxclust')
    cluster_df = pd.DataFrame({
        'asset': train.columns,
        'cluster': cluster_labels,
        'weight': [herc_weights.get(asset, 0) for asset in train.columns]
    })
    
    print(f"   Cluster distribution:")
    for cluster_id in sorted(cluster_df['cluster'].unique()):
        cluster_assets = cluster_df[cluster_df['cluster'] == cluster_id]
        total_weight = cluster_assets['weight'].sum()
        print(f"   Cluster {cluster_id}: {len(cluster_assets)} assets, total weight: {total_weight:.4f}")
        for _, row in cluster_assets.iterrows():
            print(f"     - {row['asset']}: {row['weight']:.4f}")
    
    # 8. PERFORMANCE COMPARISON
    print("\n8. Performance Comparison")
    port_ret = (test.values.flatten() @ herc_weights.values).item()
    ew_weights = np.ones(len(train.columns)) / len(train.columns)
    ew_ret = (test.values.flatten() @ ew_weights).item()
    
    print(f"   HERC CVaR Portfolio Return: {port_ret:.4%}")
    print(f"   Equal Weight Portfolio Return: {ew_ret:.4%}")
    print(f"   Difference (HERC - EW): {(port_ret - ew_ret):.4%}")
    
    # 9. RISK MEASURES
    print("\n9. Risk Measures Analysis")
    
    # Portfolio statistics
    port_volatility = np.std(train @ herc_weights.values) * np.sqrt(252)
    ew_volatility = np.std(train.mean(axis=1)) * np.sqrt(252)
    
    # CVaR Calculation
    def calculate_cvar(returns_series, alpha=0.05):
        returns_array = np.asarray(returns_series)
        var = np.percentile(returns_array, alpha * 100)
        cvar = returns_array[returns_array <= var].mean()
        return -cvar if not np.isnan(cvar) else 0
    
    port_cvar = calculate_cvar(train @ herc_weights.values)
    ew_cvar = calculate_cvar(train.mean(axis=1))
    
    print(f"   Portfolio Volatility (HERC): {port_volatility:.4f}")
    print(f"   Portfolio Volatility (EW): {ew_volatility:.4f}")
    print(f"   Portfolio CVaR 95% (HERC): {port_cvar:.6f}")
    print(f"   Portfolio CVaR 95% (EW): {ew_cvar:.6f}")
    
    # 10. DIVERSIFICATION METRICS
    print("\n10. Diversification Metrics")
    
    # Herfindahl Index
    herfindahl = np.sum(herc_weights.values ** 2)
    effective_n = 1 / herfindahl if herfindahl > 0 else len(herc_weights)
    
    # Gini Coefficient
    sorted_w = np.sort(herc_weights.values)
    n = len(sorted_w)
    gini = (2 * np.arange(1, n+1) @ sorted_w / (n * np.sum(sorted_w))) - (n+1)/n
    
    print(f"   Herfindahl Index: {herfindahl:.4f}")
    print(f"   Effective Number of Assets: {effective_n:.2f}")
    print(f"   Gini Coefficient: {gini:.4f}")
    print(f"   Maximum Weight: {herc_weights.max():.4f}")
    print(f"   Minimum Weight: {herc_weights.min():.4f}")
    
    # 11. VISUALISASI CLUSTER DENGAN CUTOFF
    print("\n11. Dendrogram dengan Cluster Cutoff")
    
    # Hitung threshold untuk pemotongan
    last_merge_distances = link[:, 2]
    if optimal_k > 0:
        cut_threshold = last_merge_distances[-optimal_k]
    else:
        cut_threshold = np.max(last_merge_distances) * 0.7
    
    plt.figure(figsize=(14, 8))
    dendrogram(link, labels=train.columns.tolist(), leaf_rotation=90, leaf_font_size=8,
               color_threshold=cut_threshold)
    
    plt.axhline(y=cut_threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Cutoff (k={optimal_k})')
    
    plt.title(f'Dendrogram dengan Cluster Cutoff (k={optimal_k})')
    plt.xlabel('Assets')
    plt.ylabel('Distance')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # 12. DETAILED GAP STATISTIC ANALYSIS
    print("\n12. Detailed Gap Statistic Analysis")
    max_clusters = min(15, len(train.columns) - 1)
    
    # Hitung Wk untuk berbagai k
    Wks_original = []
    log_Wks = []
    
    print(f"   Wk calculation for k = 1 to {max_clusters}:")
    print(f"   {'k':<3} {'Wk':<12} {'log(Wk)':<10} {'Cluster Distribution'}")
    print("   " + "-" * 60)
    
    for k in range(1, max_clusters + 1):
        labels = fcluster(link, k, criterion='maxclust')
        Wk = compute_Wk(dist_matrix, labels)
        log_Wk = np.log(Wk) if Wk > 0 else 0.0
        
        Wks_original.append(Wk)
        log_Wks.append(log_Wk)
        
        # Distribusi cluster
        unique, counts = np.unique(labels, return_counts=True)
        cluster_dist = dict(zip(unique, counts))
        
        print(f"   {k:<3} {Wk:<12.6f} {log_Wk:<10.6f} {cluster_dist}")
    
    # 13. VISUALISASI GAP STATISTIC
    print("\n13. Gap Statistic Visualization")
    
    plt.figure(figsize=(12, 8))
    
    k_values = range(1, max_clusters + 1)
    
    # Plot log(Wk)
    plt.subplot(2, 1, 1)
    plt.plot(k_values, log_Wks, 'bo-', linewidth=2, markersize=6)
    plt.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('log(Wk)')
    plt.title('Dispersion vs Number of Clusters')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot Wk values
    plt.subplot(2, 1, 2)
    plt.plot(k_values, Wks_original, 'go-', linewidth=2, markersize=6)
    plt.axvline(x=optimal_k, color='red', linestyle='--')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Wk')
    plt.title('Within-Cluster Dispersion')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'weights': herc_weights,
        'optimal_k': optimal_k,
        'portfolio_return': port_ret,
        'equal_weight_return': ew_ret,
        'link_matrix': link,
        'cluster_df': cluster_df,
        'diversification_metrics': {
            'herfindahl': herfindahl,
            'effective_n': effective_n,
            'gini': gini
        }
    }

# =======================================
# 9. EXECUTE ALL - DIPERBAIKI
# =======================================

if __name__ == "__main__":
    # Contoh penggunaan dengan data dummy
    print("="*80)
    print("RUNNING HERC ANALYSIS WITH SAMPLE DATA")
    print("="*80)
    
    # Generate sample data jika prices tidak ada
    try:
        # Jika 'prices' belum didefinisikan, buat data sampel
        if 'prices' not in globals():
            print("Generating sample crypto data...")
            n_assets = 20
            n_periods = 1000
            
            # Create synthetic asset names
            assets = [f'Crypto_{i:02d}' for i in range(1, n_assets + 1)]
            dates = pd.date_range(start='2020-01-01', periods=n_periods, freq='D')
            
            # Generate correlated returns
            np.random.seed(42)
            base_returns = np.random.randn(n_periods, 3) * 0.01
            factor_loadings = np.random.randn(n_assets, 3)
            
            returns_data = np.zeros((n_periods, n_assets))
            for i in range(n_assets):
                returns_data[:, i] = factor_loadings[i] @ base_returns.T + np.random.randn(n_periods) * 0.005
            
            returns_df = pd.DataFrame(returns_data, index=dates, columns=assets)
            prices = (1 + returns_df).cumprod()
            
            print(f"Generated sample data: {n_assets} assets, {n_periods} periods")
    
    except Exception as e:
        print(f"Error: {e}")
        # Create minimal sample data
        n_assets = 10
        n_periods = 500
        assets = [f'Asset_{i}' for i in range(n_assets)]
        dates = pd.date_range(start='2020-01-01', periods=n_periods, freq='D')
        prices = pd.DataFrame(
            np.random.randn(n_periods, n_assets).cumsum(axis=0) + 100,
            index=dates,
            columns=assets
        )
    
    # Jalankan analisis utama
    try:
        all_results, returns = main_herc_analysis(prices)
        
        if all_results:
            # Plot hasil
            plot_herc_results(all_results)
            
            # Analisis window pertama
            print("\n" + "="*80)
            print("ANALYZING FIRST WINDOW...")
            print("="*80)
            window1_analysis = analyze_first_window_herc_cvar(returns)
            
            print("\n🎯 HERC ANALYSIS COMPLETED SUCCESSFULLY!")
        else:
            print("No valid results obtained from HERC analysis.")
            
    except Exception as e:
        print(f"Error during HERC analysis: {e}")
        import traceback
        traceback.print_exc()

# =======================================
# 15. VISUALISASI 15 HARI TERAKHIR
# =======================================

def plot_last_15_days_herc(all_results):
    """Plot 15 hari terakhir untuk HERC"""
    if not all_results:
        print("No results!")
        return
    
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_results)))
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    
    for i, (combo_name, metrics) in enumerate(all_results.items()):
        label = f"{metrics['linkage_method'].upper()}-L/{metrics['risk_measure'].upper()}-R"
        
        # 1. Portfolio Value
        last_15 = metrics['final_portfolio'].tail(15)
        axes[0].plot(last_15.index, last_15.values, label=label, 
                    linewidth=2, color=colors[i], marker='o', markersize=4)
        
        # 2. Drawdown
        last_15_dd = metrics['drawdown'].tail(15)
        axes[1].plot(last_15_dd.index, last_15_dd.values, linewidth=1.5, 
                    color=colors[i], marker='s', markersize=3)
        
        # 3. Turnover
        if len(metrics['turnovers']) >= 15:
            last_15_turn = metrics['turnovers'][-15:]
            axes[2].plot(range(len(last_15_turn)), last_15_turn, linewidth=1.5,
                        color=colors[i], marker='^', markersize=3)
    
    axes[0].set_title('Portfolio Value - 15 Hari Terakhir', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Value')
    axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[0].grid(True, alpha=0.3)
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    
    axes[1].set_title('Drawdown - 15 Hari Terakhir', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Drawdown')
    axes[1].grid(True, alpha=0.3)
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    axes[2].set_title('Turnover - 15 Periode Terakhir', fontsize=14, fontweight='bold')
    axes[2].set_ylabel('Turnover')
    axes[2].set_xlabel('Rolling Window')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Jalankan visualisasi 15 hari terakhir
if all_results:
    plot_last_15_days_herc(all_results)
else:
    print("No HERC results available for 15-day analysis!")

print("\n🎯 HERC Analysis Completed!")

def extract_herc_weights_table(rolling_results, returns, window_train=254):
    """
    Mengubah list weights_all (Series) menjadi DataFrame time × asset
    """
    weights_list = rolling_results['weights_all']

    if len(weights_list) == 0:
        return None

    # index waktu rebalancing
    rebalance_dates = returns.index[window_train : window_train + len(weights_list)]

    # gabungkan Series → DataFrame
    weights_df = pd.DataFrame(weights_list)
    weights_df.index = rebalance_dates

    # pastikan kolom konsisten
    weights_df = weights_df.fillna(0.0)

    return weights_df


def extract_all_herc_weights_all_combinations(
    returns,
    linkage_methods,
    risk_measures,
    precomputed_clusters,
    alpha=0.05
):
    all_weights_tables = {}

    for linkage_method in linkage_methods:
        for risk_measure in risk_measures:
            combo_name = f"{linkage_method}_{risk_measure}"
            print(f"Extracting weights for {combo_name}...")

            rolling_results = run_rolling_herc_consistent(
                returns,
                linkage_method=linkage_method,
                risk_measure=risk_measure,
                alpha=alpha,
                precomputed_clusters=precomputed_clusters
            )

            weights_df = extract_herc_weights_table(
                rolling_results,
                returns
            )

            if weights_df is not None:
                all_weights_tables[combo_name] = weights_df
                print(f"  ✓ {combo_name}: {weights_df.shape}")
            else:
                print(f"  ✗ {combo_name}: empty")

    return all_weights_tables

linkage_methods = ['ward', 'complete', 'average', 'single']
risk_measures = ['variance', 'cvar', 'cdar']

all_herc_weights_tables = extract_all_herc_weights_all_combinations(
    returns=returns,
    linkage_methods=linkage_methods,
    risk_measures=risk_measures,
    precomputed_clusters=precomputed_clusters,
    alpha=0.05
)
all_herc_weights_tables.keys()